In [ ]:
# =============================================================================
# Zelle 01 – Setup: Pfade, Konfiguration, Stil, Datenbasis prüfen
# =============================================================================
# Zweck: Gemeinsame Grundlage für alle EDA-Zellen.
# Datenbasis: ausschließlich der Snapshot in data/raw (Prinzip P7).
#   Einzige dokumentierte Ausnahme: Zelle 10 liest Tags nur lesend vom Stick
#   (Entscheidung 07.11.2026; 126 GB Kopie unverhältnismäßig).
# Artefakte: Namensschema 02_<Zelle>_<slug>; Tabellen über tabelle_speichern(),
#   Grafiken über figur_speichern() – beide prüfen das Schema.
# =============================================================================

import sys
from pathlib import Path

import pandas as pd
import yaml

from playlist_agent.viz_config import ARTEFAKT_MUSTER, FARBEN, figur_speichern, stil_aktivieren

# --- Projektpfade (gleiche Logik wie Notebook 01) ----------------------------
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
assert PROJECT_ROOT.name == "_MyMSIT_AI-Agent_repair-playlist", PROJECT_ROOT

RAW = PROJECT_ROOT / "data" / "raw"
FIG = PROJECT_ROOT / "reports" / "figures"
TAB = PROJECT_ROOT / "reports" / "tables"
ERR = PROJECT_ROOT / "reports" / "errors"      # privat (gitignored)

GOLD = PROJECT_ROOT / "data" / "gold"           # privat: Gold-Labels, Eingabedateien


def lese_eingabe(name: str) -> list[str]:
    """Liest eine private Eingabedatei aus data/gold (1 Wert je Zeile).

    Echte Pfade/Titel stehen nie im Code (strenge Regel, Entscheidung 07.11.2026),
    sondern in diesen ignorierten Dateien.
    """
    pfad = GOLD / name
    assert pfad.exists(), f"Private Eingabedatei fehlt: data/gold/{name}"
    return [z.strip() for z in pfad.read_text(encoding="utf-8").splitlines() if z.strip()]

cfg = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
stil_aktivieren()


def tabelle_speichern(df: pd.DataFrame, name: str, ordner: Path = TAB) -> Path:
    """CSV nach dem Namensschema <NB>_<Zelle>_<slug> speichern.

    Gleiche Prüfung wie figur_speichern(). Vorerst hier im Notebook;
    Umzug nach src/ in Phase 3, sobald sie mehrfach gebraucht wird.
    """
    if not ARTEFAKT_MUSTER.match(name):
        raise ValueError(f"Name '{name}' entspricht nicht <NB>_<Zelle>_<slug>.")
    pfad = ordner / f"{name}.csv"
    df.to_csv(pfad, index=False, encoding="utf-8")
    return pfad


# --- Datenbasis prüfen: Snapshot vollständig? ---------------------------------
df_manifest = pd.read_csv(RAW / "snapshot_manifest.csv")
df_index = pd.read_csv(RAW / "index" / "musik_index.csv")

print(f"Python  : {sys.version.split()[0]}")
print(f"pandas  : {pd.__version__}")
print(f"Snapshot: {len(df_manifest)} Dateien, davon bitgenau {df_manifest['identisch'].sum()}")
print(df_manifest.groupby("quelle").size().to_string())
print(f"Index   : {len(df_index)} Dateien")
print(f"\nDatentyp Spalte 'pfad_relativ' im Index: {df_index['pfad_relativ'].dtype}")

In [ ]:
# =============================================================================
# Zelle 02 – Alle Crates parsen: Einträge, Verteilung, Integrität, pandas-3-Check
# =============================================================================
# Zweck: Vollständige Tabelle aller Crate-Einträge als Grundlage der EDA.
# Format (Vorbefund): TLV-Blöcke, 4-Byte-Tag + 4-Byte-Länge Big-Endian,
#   Strings UTF-16-BE; Track = 'otrk', darin Pfad = 'ptrk'.
# Integritätsprüfung: Datei muss exakt bis zum letzten Byte aufgehen.
# pandas-3-Check: bleibt die Unicode-Normalform (NFC/NFD) in Spalten erhalten?
# Datenschutz: Einzeltabelle mit Crate-Namen/Pfaden privat in data/interim;
#   ins Repo nur Verteilungsstatistik und Grafik.
# Prototyp im Notebook, Umzug nach src/playlist_agent/parsers/ in Phase 3.
# =============================================================================

import unicodedata
import matplotlib.pyplot as plt


def parse_crate(pfad: Path) -> dict:
    """Liest eine Serato-Crate; liefert Version, Pfadliste und Integritätsstatus."""
    b = pfad.read_bytes()
    i, version, pfade = 0, None, []
    while i + 8 <= len(b):
        tag, n = b[i:i + 4], int.from_bytes(b[i + 4:i + 8], "big")
        wert = b[i + 8:i + 8 + n]
        i += 8 + n
        if tag == b"vrsn":
            version = wert.decode("utf-16-be")
        elif tag == b"otrk":                      # Track-Block: darin 'ptrk' suchen
            j = 0
            while j + 8 <= len(wert):
                t2, n2 = wert[j:j + 4], int.from_bytes(wert[j + 4:j + 8], "big")
                if t2 == b"ptrk":
                    pfade.append(wert[j + 8:j + 8 + n2].decode("utf-16-be"))
                j += 8 + n2
    return {"version": version, "pfade": pfade, "vollstaendig": i == len(b)}


# --- Alle Crates parsen ---------------------------------------------------------
crate_dateien = sorted((RAW / "crates").glob("*.crate"))
eintraege, crates = [], []
for f in crate_dateien:
    try:
        r = parse_crate(f)
        crates.append({"crate": f.stem, "version": r["version"],
                       "n_tracks": len(r["pfade"]), "vollstaendig": r["vollstaendig"],
                       "fehler": None})
        eintraege += [{"crate": f.stem, "position": k, "pfad": p}
                      for k, p in enumerate(r["pfade"], start=1)]
    except Exception as e:                        # einzelne Crate darf nicht alles stoppen
        crates.append({"crate": f.stem, "version": None, "n_tracks": 0,
                       "vollstaendig": False, "fehler": repr(e)})

df_crates = pd.DataFrame(crates)                  # 1 Zeile je Crate
df_eintraege = pd.DataFrame(eintraege)            # 1 Zeile je Crate-Eintrag
df_eintraege.to_csv(PROJECT_ROOT / "data/interim/02_02_crate_eintraege.csv",
                    index=False, encoding="utf-8")   # privat

# --- Integrität -------------------------------------------------------------------
print(f"Crates gelesen       : {len(df_crates)}")
print(f"Versionen            : {df_crates['version'].value_counts(dropna=False).to_dict()}")
print(f"vollständig gelesen  : {df_crates['vollstaendig'].sum()} / {len(df_crates)}")
print(f"Lesefehler           : {df_crates['fehler'].notna().sum()}")
print(f"leere Crates (0 Tr.) : {(df_crates['n_tracks'] == 0).sum()}")

# --- Umfang ------------------------------------------------------------------------
print(f"\nEinträge gesamt      : {len(df_eintraege)}")
print(f"eindeutige Pfade     : {df_eintraege['pfad'].nunique()}")
stat_tracks = df_crates["n_tracks"].describe(percentiles=[.25, .5, .75, .9]).round(1)
print("\nTracks je Crate:\n" + stat_tracks.to_string())

# --- pandas-3-Check: Normalform bleibt erhalten? -----------------------------------
n_nfd_python = sum(unicodedata.normalize("NFC", p) != p for p in df_eintraege["pfad"].tolist())
n_nfd_pandas = (df_eintraege["pfad"] != df_eintraege["pfad"].str.normalize("NFC")).sum()
print(f"\nNFD-Pfade (reines Python): {n_nfd_python} | (pandas-Vergleich): {n_nfd_pandas} "
      f"| Datentyp: {df_eintraege['pfad'].dtype}")

# --- Artefakt 1: Verteilungsstatistik (public, ohne Namen) --------------------------
df_tracks_je_crate = (stat_tracks.rename("tracks_je_crate").reset_index()
                      .rename(columns={"index": "kennzahl"}))
tabelle_speichern(df_tracks_je_crate, "02_02_tracks_je_crate")

# --- Artefakt 2: Histogramm (log-Achse wegen starker Rechtsschiefe) ---------------
# Begründung: Median 13 vs. Max 1330 -> lineare Achse drückt 140 von 167 Crates
#   in den ersten Balken. Log-Klassen geben jeder Größenordnung gleich viel Platz.
import numpy as np

klassen = np.logspace(0, np.log10(df_crates["n_tracks"].max()), 25)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_crates["n_tracks"], bins=klassen, color=FARBEN["gold"],
        edgecolor=FARBEN["hintergrund"])
ax.axvline(df_crates["n_tracks"].median(), color=FARBEN["blau"], lw=2,
           label=f"Median = {df_crates['n_tracks'].median():.0f}")
ax.set_xscale("log")
ax.set_xticks([1, 3, 10, 30, 100, 300, 1000])
ax.set_xticklabels(["1", "3", "10", "30", "100", "300", "1000"])
ax.set_xlabel("Tracks je Crate (logarithmische Achse)")
ax.set_ylabel("Anzahl Crates")
ax.set_title(f"Größenverteilung der {len(df_crates)} Serato-Crates")
ax.legend()
figur_speichern(fig, "02_02_tracks_je_crate", FIG)
plt.show()

In [ ]:
# =============================================================================
# Zelle 03 – Zeichenkodierung der Crate-Pfade: Fallstudie, Klassen, Wirkung, je Crate
# =============================================================================
  # Anlass: Stichprobe mit Umlaut – Editor zeigt in der Crate "a" statt "ä".
  #   Lücken = UTF-16-Darstellung im Editor (00-Bytes), keine echten Leerzeichen.
#   Offen: Ist das "a" + kombinierendes Trema (NFD) oder ein verlorener Umlaut?
# Methode:
#   A Fallstudie Codepoints | B Klassen ASCII/NFC/NFD je eindeutigem Pfad
#   C Treffer im Index: exakt -> erst nach NFC -> erst nach Akzent-Entfernung
#   D NFD je Crate (Prüfung der auffälligen Zahl 656)
# Datenschutz: Zählstatistik public (reports/tables), Einzelpfade privat (reports/errors).
# =============================================================================


def akzente_entfernen(s: str) -> str:
    """'Väth' -> 'vath': zerlegt Zeichen (NFKD), entfernt Akzente, Kleinschreibung."""
    return "".join(c for c in unicodedata.normalize("NFKD", s)
                   if not unicodedata.combining(c)).lower()


def klasse(p: str) -> str:
    if all(ord(c) < 128 for c in p):
        return "ASCII"
    return "NFD" if unicodedata.normalize("NFC", p) != p else "NFC mit Sonderzeichen"

# --- A: Fallstudie (Suchtext aus privater Eingabedatei) ---------------------------
print("=== A: Fallstudie ===")
for suchtext in lese_eingabe("02_03_fallstudie.txt"):
    for quelle, serie in [("Crate", df_eintraege["pfad"]), ("Index", df_index["pfad_relativ"])]:
        for p in serie[serie.str.contains(suchtext, regex=False)].unique():
            zeichen = [f"'{c}' U+{ord(c):04X}" for c in p if ord(c) > 127]
            print(f"{quelle}: {p!r}\n   Sonderzeichen: {zeichen} | Klasse: {klasse(p)}")

# --- B + C: Klassen und Treffer je eindeutigem Crate-Pfad ---------------------------
idx_exakt = set(df_index["pfad_relativ"])
idx_nfc = {unicodedata.normalize("NFC", p) for p in idx_exakt}
idx_akzentfrei = {akzente_entfernen(p) for p in idx_exakt}

df_pfade = pd.DataFrame({"pfad": df_eintraege["pfad"].unique()})
df_pfade["klasse"] = df_pfade["pfad"].map(klasse)
df_pfade["treffer"] = np.select(
    [df_pfade["pfad"].isin(idx_exakt),
     df_pfade["pfad"].map(lambda p: unicodedata.normalize("NFC", p) in idx_nfc),
     df_pfade["pfad"].map(lambda p: akzente_entfernen(p) in idx_akzentfrei)],
    ["1 exakt", "2 erst nach NFC", "3 erst ohne Akzente"],
    default="4 kein Treffer")

df_klasse_treffer = pd.crosstab(df_pfade["klasse"], df_pfade["treffer"],
                                margins=True, margins_name="Summe")
print("\n=== B+C: eindeutige Crate-Pfade nach Klasse und Treffer im Index ===")
print(df_klasse_treffer.to_string())
tabelle_speichern(df_klasse_treffer.reset_index(), "02_03_kodierung_klasse_treffer")

# Einzelpfade der schwierigen Fälle privat sichern (für Zelle 05)
df_pfade[df_pfade["treffer"].isin(["3 erst ohne Akzente", "4 kein Treffer"])] \
    .to_csv(ERR / "02_03_pfade_ohne_direkten_treffer.csv", index=False, encoding="utf-8")

# --- D: NFD je Crate ----------------------------------------------------------------
df_eintraege["nfd"] = df_eintraege["pfad"].map(lambda p: klasse(p) == "NFD")
df_nfd_je_crate = (df_eintraege.groupby("crate")["nfd"]
                   .agg(n_eintraege="size", n_nfd="sum").reset_index())
n_crates_mit_nfd = (df_nfd_je_crate["n_nfd"] > 0).sum()
print(f"\n=== D: NFD je Crate ===")
print(f"NFD-Einträge gesamt     : {df_eintraege['nfd'].sum()}  "
      f"(eindeutige NFD-Pfade: {(df_pfade['klasse'] == 'NFD').sum()})")
print(f"Crates mit NFD-Einträgen: {n_crates_mit_nfd} von {len(df_nfd_je_crate)}")
print("Top 5 Crates nach NFD-Einträgen:")
print(df_nfd_je_crate.sort_values("n_nfd", ascending=False).head(5).to_string(index=False))

# --- Grafik: Treffer je Klasse (eindeutige Pfade) ------------------------------------
# Farbbedeutung: Grün = direkt gelöst, Gold = Kernbefund (erst nach NFC),
#   Blau = Akzent verloren, Grau = kein Treffer.
df_plot = df_klasse_treffer.drop(index="Summe", columns="Summe")
farben = {"1 exakt": FARBEN["gruen"], "2 erst nach NFC": FARBEN["gold"],
          "3 erst ohne Akzente": FARBEN["blau"], "4 kein Treffer": FARBEN["text_gedimmt"]}
fig, ax = plt.subplots(figsize=(10, 3.5))
links = np.zeros(len(df_plot))
for spalte in df_plot.columns:
    ax.barh(df_plot.index, df_plot[spalte], left=links, color=farben[spalte], label=spalte)
    links += df_plot[spalte].to_numpy()
ax.set_xlabel("eindeutige Crate-Pfade")
ax.set_title("Crate-Pfade nach Zeichenkodierung und Treffer im Index")
ax.legend(loc="lower right", fontsize=10)
figur_speichern(fig, "02_03_kodierung_klasse_treffer", FIG)
plt.show()

In [ ]:
# =============================================================================
# Zelle 04 – Evidenz der Trefferstufen: Index-Format, Beispiele je Stufe,
#            Stichprobe für unabhängigen foobar-Test
# =============================================================================
# Zweck: Behauptungen aus Zelle 03 an konkreten Pfaden belegen (erst Evidenz,
#   dann Aussage) und eine Stichprobe für eine UNABHÄNGIGE Referenz erzeugen:
#   Songs der Stichprobe werden von Hand in foobar eingefügt und als Playlist
#   gespeichert; der von foobar geschriebene Pfad wird in Zelle 05 mit Index
#   und Crate verglichen.
# Reproduzierbar: Zufallsauswahl mit festem Startwert (SEED = 42).
# Datenschutz: Pfade nur als Notebook-Ausgabe (nbstripout entfernt sie beim
#   Commit); Stichprobenliste privat in data/gold/.
# =============================================================================

SEED = 42
STICK_LW = "D:"          # Laufwerk des Sticks auf dem Asus (Zelle 06, Notebook 01)


def sonderzeichen(p: str) -> list[str]:
    """Alle Nicht-ASCII-Zeichen mit Codepoint, z. B. ['ä' U+00E4]."""
    return [f"'{c}' U+{ord(c):04X}" for c in p if ord(c) > 127]


def als_windows_pfad(p: str) -> str:
    """'Musik/A/x.mp3' -> 'D:\\Musik\\A\\x.mp3' (zum Suchen im Explorer/foobar)."""
    return STICK_LW + "\\" + p.replace("/", "\\")


# Zuordnung NFC-Form -> tatsächlicher Index-Pfad (für Stufe 2: welcher Pfad rechts?)
idx_nfc_zu_pfad = {unicodedata.normalize("NFC", p): p for p in df_index["pfad_relativ"]}


def index_partner(p: str, stufe: str) -> str | None:
    if stufe == "1 exakt":
        return p
    if stufe == "2 erst nach NFC":
        return idx_nfc_zu_pfad.get(unicodedata.normalize("NFC", p))
    return None


def zeige(links: str, stufe: str) -> None:
    rechts = index_partner(links, stufe)
    print(f"  Stufe : {stufe}")
    print(f"  links (Crate): {links!r}")
    print(f"        Sonderzeichen: {sonderzeichen(links)}")
    print(f"  rechts (Index): {rechts!r}")
    if rechts:
        print(f"        Sonderzeichen: {sonderzeichen(rechts)}")
        print(f"        Zeichen-identisch: {links == rechts}")
    print()


# --- A: Format eines Index-Pfads ("rechts") ----------------------------------------
print("=== A: So sieht ein Pfad im Index aus (3 Zufallsbeispiele) ===")
for p in df_index["pfad_relativ"].sample(3, random_state=SEED):
    print(f"  {p!r}")
print("  Format: relativ zur Stick-Wurzel, ohne Laufwerk, Trenner '/', "
      "gleiche Form wie in den Crates.\n")

# --- B: Nutzer-Stichprobe (ohne Umlaut): Suche in Crates UND Index ---------------
print("=== B: Beispiele aus der Nutzer-Stichprobe ===")
for suchtext in lese_eingabe("02_04_nutzer_stichprobe.txt"):
    funde_crate = df_pfade[df_pfade["pfad"].str.contains(suchtext, regex=False)]
    funde_index = df_index[df_index["pfad_relativ"].str.contains(suchtext, regex=False)]
    print(f"Suche '{suchtext}': {len(funde_crate)} Crate-Pfad(e), "
          f"{len(funde_index)} Index-Pfad(e)")
    for _, z in funde_crate.iterrows():
        zeige(z["pfad"], z["treffer"])
    if funde_crate.empty:
        for p in funde_index["pfad_relativ"]:
            print(f"  nur im Index (in keiner Crate): {p!r}\n")

# --- C: je Stufe 3 Zufallsbeispiele ---------------------------------------------------
print("=== C: je Trefferstufe 3 Zufallsbeispiele ===")
for stufe in ["1 exakt", "2 erst nach NFC", "3 erst ohne Akzente", "4 kein Treffer"]:
    gruppe = df_pfade[df_pfade["treffer"] == stufe]
    print(f"--- {stufe}: {len(gruppe)} Pfade insgesamt ---")
    if gruppe.empty:
        print("  (keine Fälle – Befund, kein fehlendes Beispiel)\n")
        continue
    for p in gruppe["pfad"].sample(min(3, len(gruppe)), random_state=SEED):
        zeige(p, stufe)


# --- D: Stichprobe für den foobar-Test ----------------------------------------------
# 4 x Stufe 1 ASCII, 3 x Stufe 1 mit Sonderzeichen, 3 x Stufe 2 (NFD),
# 3 x Stufe 4 (Titel wird von Hand gesucht: gefunden -> in foobar aufnehmen,
#   nicht gefunden -> als "fehlt" notieren).
plan = [("1 exakt", "ASCII", 4), ("1 exakt", "NFC mit Sonderzeichen", 3),
        ("2 erst nach NFC", "NFD", 3), ("4 kein Treffer", None, 3)]
teile = []
for stufe, kl, n in plan:
    g = df_pfade[(df_pfade["treffer"] == stufe) & ((df_pfade["klasse"] == kl) if kl else True)]
    teile.append(g.sample(n, random_state=SEED))
df_stichprobe = pd.concat(teile, ignore_index=True)
df_stichprobe["index_pfad"] = [index_partner(p, s) for p, s in
                               zip(df_stichprobe["pfad"], df_stichprobe["treffer"])]
df_stichprobe["suchen_unter"] = [als_windows_pfad(ip) if pd.notna(ip) else
                                 "TITEL VON HAND SUCHEN: " + p.rsplit("/", 1)[-1]
                                 for ip, p in zip(df_stichprobe["index_pfad"],
                                                  df_stichprobe["pfad"])]
df_stichprobe.insert(0, "nr", range(1, len(df_stichprobe) + 1))
df_stichprobe.to_csv(PROJECT_ROOT / "data/gold/02_04_stichprobe_foobar.csv",
                     index=False, encoding="utf-8")       # privat

print("=== D: Stichprobe für den foobar-Test (gespeichert: data/gold/02_04_stichprobe_foobar.csv) ===")
print(df_stichprobe[["nr", "treffer", "klasse", "suchen_unter"]].to_string(index=False))



In [ ]:
# =============================================================================
# Zelle 05 – Unabhängige Referenz: foobar-Pfade vs. Index vs. Crate; Encoding
# =============================================================================
# Zweck: Die von foobar selbst geschriebenen Pfade (Stichprobe aus Zelle 04)
#   dienen als unabhängige Referenz für die Trefferstufen 1/2. Zusätzlich:
#   Wie kodiert foobar Umlaute in .m3u und .m3u8 (Bytes, BOM, Zeilenende)?
# Stufe-4-Fälle: Ergebnis der manuellen Suche als Pfad oder "fehlt".
# Ergebnis: erste Gold-Labels (data/gold, privat) – Ziel-Datei oder "fehlt".
# Zuordnung foobar-Eintrag -> Stichprobe über NFC-gleichen Referenzpfad,
#   nicht über die Reihenfolge (robust gegen Einfügefehler).
# =============================================================================

# Manuelle Suche der Stufe-4-Fälle (private Eingabedatei: Pfad relativ zur
# Stick-Wurzel oder "fehlt"); GOLD ist in Zelle 01 definiert
df_handsuche = pd.read_csv(GOLD / "02_05_handsuche.csv")
HANDSUCHE = dict(zip(df_handsuche["nr"].astype(int), df_handsuche["ergebnis"]))


def lese_playlist(pfad: Path) -> dict:
    """Liest eine foobar-Playlist byte-genau: BOM, Zeilenende, Encoding, Pfade."""
    roh = pfad.read_bytes()
    bom = roh.startswith(b"\xef\xbb\xbf")
    try:
        text, encoding = roh.decode("utf-8-sig"), "utf-8"
    except UnicodeDecodeError:
        text, encoding = roh.decode("cp1252"), "cp1252 (ANSI)"
    zeilen = text.splitlines()
    return {"roh": roh, "bom": bom, "encoding": encoding,
            "crlf": b"\r\n" in roh,
            "kommentare": [z for z in zeilen if z.startswith("#")],
            "pfade": [z for z in zeilen if z.strip() and not z.startswith("#")]}


def zu_relativ(windows_pfad: str) -> str:
    """'D:\\Musik\\A\\x.mp3' -> 'Musik/A/x.mp3' (Index-Form)."""
    return windows_pfad.split(":\\", 1)[-1].replace("\\", "/")


nfc = lambda s: unicodedata.normalize("NFC", s)

# --- Referenzpfad je Stichprobe: Index-Partner (Stufe 1/2) bzw. Handsuche (Stufe 4)
df_stichprobe = pd.read_csv(GOLD / "02_04_stichprobe_foobar.csv")
df_stichprobe["referenz_pfad"] = df_stichprobe["index_pfad"]
for nr, ergebnis in HANDSUCHE.items():
    if ergebnis != "fehlt":
        df_stichprobe.loc[df_stichprobe["nr"] == nr, "referenz_pfad"] = ergebnis

for endung in ["m3u", "m3u8"]:
    pl = lese_playlist(GOLD / f"02_05_foobar_stichprobe_asus.{endung}")
    print(f"=== foobar .{endung} ===")
    print(f"  BOM: {pl['bom']} | Encoding (erkannt): {pl['encoding']} | CRLF: {pl['crlf']}")
    print(f"  Kommentar-/Kopfzeilen: {pl['kommentare'][:3]}")
    print(f"  Einträge: {len(pl['pfade'])}")

    # Byte-Evidenz: Wie stehen Umlaute in der Datei?
    for wort in ["Väth", "Planisphäre"]:
        for kodierung, b in [("UTF-8", wort.encode("utf-8")), ("cp1252", wort.encode("cp1252"))]:
            if b in pl["roh"]:
                print(f"  '{wort}' als {kodierung}-Bytes gefunden: {b.hex(' ')}")

    # Vergleich je foobar-Eintrag
    zeilen = []
    for fp in pl["pfade"]:
        rel = zu_relativ(fp)
        treffer = df_stichprobe[df_stichprobe["referenz_pfad"].fillna("").map(nfc) == nfc(rel)]
        z = treffer.iloc[0] if len(treffer) else None
        zeilen.append({
            "nr": int(z["nr"]) if z is not None else None,
            "foobar_normalform": "NFD" if nfc(rel) != rel else "NFC",
            "gleich_referenz_exakt": z is not None and rel == z["referenz_pfad"],
            "gleich_crate_exakt": z is not None and rel == z["pfad"],
            "gleich_crate_nach_nfc": z is not None and nfc(rel) == nfc(z["pfad"]),
        })
    df_vergleich = pd.DataFrame(zeilen)
    print(df_vergleich.to_string(index=False))
    print()

# --- Gold-Labels der Stichprobe ------------------------------------------------
df_gold_stichprobe = df_stichprobe[["nr", "treffer", "klasse", "pfad", "referenz_pfad"]].copy()
df_gold_stichprobe["gold_ziel"] = df_gold_stichprobe["referenz_pfad"].fillna("fehlt")
df_gold_stichprobe["gold_quelle"] = ["Handsuche" if n in HANDSUCHE else "foobar"
                                     for n in df_gold_stichprobe["nr"]]
df_gold_stichprobe.to_csv(GOLD / "02_05_gold_stichprobe.csv", index=False, encoding="utf-8")
print(df_gold_stichprobe[["nr", "treffer", "gold_quelle", "gold_ziel"]].to_string(index=False))

### Erkenntnisse Zellen 02–05: Crates, Zeichenkodierung, unabhängige Referenz

**Evidenz**

| Prüfung | Ergebnis |
|---|---|
| Integrität | 167/167 Crates vollständig bis zum letzten Byte gelesen, 0 Fehler, 1 Version |
| Umfang | 7.915 Einträge, 2.754 eindeutige Pfade (Ø 2,9 Crates je Pfad); 52,6 % der 5.236 Audiodateien haben Crate-Bezug |
| Verteilung | Tracks je Crate: Median 13, Mittelwert 47,4, Max 1.330 (stark rechtsschief) |
| pandas 3 | Datentyp `str`; NFD-Zählung Python = pandas = 656 → Normalform bleibt erhalten |
| Trefferstufen (alle 2.754 Pfade) | 1 exakt 2.075 (75,3 %) · 2 erst nach NFC 275 (10,0 %) · 3 erst ohne Akzente 0 · 4 kein Treffer 404 (14,7 %) |
| Evidenz je Stufe (Zelle 04) | Stufe 1: 3/3 zeichen-identisch, auch mit Umlaut in NFC; Stufe 2: 3/3 mit U+0308 (Crate) vs. U+00E4/U+00FC (Stick) |
| Unabhängige Referenz (Zelle 05) | foobar-Pfade 11/11 identisch mit Referenzpfad; NFD-Fälle 3/3 erst nach NFC gleich; verschobener Fall 1/1 weder exakt noch nach NFC gleich |
| Kodierung foobar | `.m3u`: cp1252, ohne BOM, `ä` = `e4` · `.m3u8`: UTF-8 mit BOM, Kopfzeile `#`, `ä` = `c3 a4` · foobar schreibt NFC |
| Gold-Labels | 13 Stichproben: 10 per foobar bestätigt, 1 verschoben (per Handsuche gefunden), 2 fehlend |

**Aussagen (aus der Evidenz abgeleitet)**

- 85,3 % der eindeutigen Crate-Pfade sind mit den Stufen 1 und 2 deterministisch auflösbar.
- NFD betrifft 300 Pfade in 72 von 167 Crates; Stufe 2 (NFC-Normalisierung beider Seiten) ist zwingend.
- Unter Pfaden, die bis auf Akzente identisch sind, gibt es keinen verlorenen Umlaut (Stufe 3 = 0).
- Die 404 Fehltreffer sind überwiegend kein Kodierungsproblem: Fehlquote ASCII 18,8 % gegenüber NFC 7,5 % und NFD 8,3 %.
- Das Ausgabeformat bestimmt die Kodierung zwingend (`.m3u` = cp1252, `.m3u8` = UTF-8).

**Grenzen und offene Punkte**

- 11 foobar-Vergleiche belegen den Mechanismus, nicht die Fehlerfreiheit (Rule of Three: Fehlerrate < 27 %).
- Nicht erfasst: verlorene Umlaute in zusätzlich verschobenen Pfaden, Mojibake-Dateinamen (20 auf dem Stick) → Zelle 06.
- 25 NFD-Pfade ohne Treffer auch nach NFC → Ursache in Zelle 06.
- Ursache der Mischung NFC/NFD in den Crates offen (alle Crates am Mac erstellt).

**Entscheidungen**

- Stufe 4.2 = NFC-Normalisierung beider Seiten.
- Ausgabeformat (`.m3u` ANSI vs. `.m3u8` UTF-8) wird in Phase 3 datenbasiert entschieden.

In [ ]:
# =============================================================================
# Zelle 06 – Fehltreffer (Stufe 4) nach Namensabgleich im Index
# =============================================================================
# Rechenvorschrift: Schlüssel k(p) = Kleinschreibung(NFC(Dateiname(p))).
#   I = alle echten Audiodateien im Index (Endung laut config, ohne AppleDouble).
#   Je Crate-Pfad p ohne Treffer in Stufe 1-3, erste zutreffende Kategorie:
#   a |{q in I : k(q) = k(p)}| = 1          (Name genau 1x im Index, anderer Ordner)
#   b |{q in I : k(q) = k(p)}| > 1          (Name mehrfach im Index)
#   c k(p) = k(Mojibake-Reparatur(q)) für q in I  (Name erst nach Reparatur)
#   d Name nicht in I, Ordner(p) im Index vorhanden
#   e Name nicht in I, Ordner(p) im Index nicht vorhanden
#   Zusatzmerkmal: Ordner(p) fehlt, aber ein Ordner gleichen Namens existiert anderswo.
# Grenze: Namensgleichheit belegt KEINE Inhaltsgleichheit (Audio). Inhaltsbeleg
#   erst über Metadaten (z. B. Länge aus Serato database V2, Zelle 07).
# Mögliche Ursachen (Hypothesen, nicht geprüft): a/b verschoben; d/e gelöscht,
#   umbenannt oder umbenannt und verschoben.
# Validierung: Abgleich mit Gold-Labels der Handsuche (Zelle 05).
# Datenschutz: Zählstatistik public; Einzelpfade privat in reports/errors.
# =============================================================================

KAT_A = "a Name 1x im Index, anderer Ordner"
KAT_B = "b Name mehrfach im Index"
KAT_C = "c Name erst nach Mojibake-Reparatur"
KAT_D = "d Name nicht im Index, Ordner vorhanden"
KAT_E = "e Name nicht im Index, Ordner nicht vorhanden"


def schluessel(s: str) -> str:
    return unicodedata.normalize("NFC", s).lower()


def mojibake_reparieren(s: str) -> str:
    """'FlÃ¼gel' -> 'Flügel': als cp1252 kodieren, als UTF-8 lesen; sonst unverändert."""
    try:
        return s.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s


dateiname = lambda p: p.rsplit("/", 1)[-1]
ordner = lambda p: p.rsplit("/", 1)[0]

# --- Nachschlagetabellen aus dem Index -------------------------------------------
audio = set(cfg["index"]["audio_endungen"])
df_audio = df_index[df_index["endung"].isin(audio) & ~df_index["appledouble"]].copy()
df_audio["name_key"] = df_audio["pfad_relativ"].map(lambda p: schluessel(dateiname(p)))
df_audio["name_key_rep"] = df_audio["pfad_relativ"].map(
    lambda p: schluessel(mojibake_reparieren(dateiname(p))))

name_zu_pfaden = df_audio.groupby("name_key")["pfad_relativ"].apply(list).to_dict()
moji_zu_pfaden = (df_audio[df_audio["name_key_rep"] != df_audio["name_key"]]
                  .groupby("name_key_rep")["pfad_relativ"].apply(list).to_dict())
ordner_im_index = {schluessel(ordner(p)) for p in df_index["pfad_relativ"]}
ordnernamen_im_index = {schluessel(dateiname(ordner(p))) for p in df_index["pfad_relativ"]}

print(f"Audiodateien im Index (I): {len(df_audio)} | "
      f"davon mit reparierbarem Mojibake-Namen: {(df_audio['name_key_rep'] != df_audio['name_key']).sum()}")

# --- Kategorisierung ---------------------------------------------------------------
df_rest = df_pfade[df_pfade["treffer"] == "4 kein Treffer"].copy()


def kategorisiere(p: str) -> tuple[str, list[str]]:
    k = schluessel(dateiname(p))
    if k in name_zu_pfaden:
        kand = name_zu_pfaden[k]
        return (KAT_A if len(kand) == 1 else KAT_B), kand
    if k in moji_zu_pfaden:
        return KAT_C, moji_zu_pfaden[k]
    return (KAT_D if schluessel(ordner(p)) in ordner_im_index else KAT_E), []


erg = df_rest["pfad"].map(kategorisiere)
df_rest["kategorie"] = erg.map(lambda t: t[0])
df_rest["kandidaten"] = erg.map(lambda t: t[1])
df_rest["n_kandidaten"] = df_rest["kandidaten"].map(len)
df_rest["ordnername_anderswo"] = [
    schluessel(ordner(p)) not in ordner_im_index
    and schluessel(dateiname(ordner(p))) in ordnernamen_im_index
    for p in df_rest["pfad"]]

# --- Ergebnis 1: Kategorie x Kodierungsklasse ---------------------------------------
df_kat_klasse = pd.crosstab(df_rest["kategorie"], df_rest["klasse"],
                            margins=True, margins_name="Summe")
print("\n=== Fehltreffer (eindeutige Pfade) nach Kategorie und Kodierungsklasse ===")
print(df_kat_klasse.to_string())
tabelle_speichern(df_kat_klasse.reset_index(), "02_06_fehltreffer_kategorie_klasse")

# --- Ergebnis 2: Zusatzmerkmal "Ordnername anderswo" je Kategorie --------------------
df_kat_ordner = pd.crosstab(df_rest["kategorie"], df_rest["ordnername_anderswo"],
                            margins=True, margins_name="Summe")
print("\n=== Ordner am Crate-Ort fehlt, gleichnamiger Ordner existiert anderswo (True) ===")
print(df_kat_ordner.to_string())
tabelle_speichern(df_kat_ordner.reset_index(), "02_06_fehltreffer_ordnername_anderswo")

df_rest.to_csv(ERR / "02_06_fehltreffer_kategorien.csv", index=False, encoding="utf-8")  # privat

# --- Evidenz: je Kategorie 2 Beispiele -------------------------------------------
print("\n=== Beispiele je Kategorie ===")
for kat, g in df_rest.groupby("kategorie"):
    print(f"--- {kat}: {len(g)} ---")
    for _, z in g.sample(min(2, len(g)), random_state=SEED).iterrows():
        print(f"  Crate     : {z['pfad']!r}")
        print(f"  Kandidaten: {z['kandidaten'][:3]}{' …' if z['n_kandidaten'] > 3 else ''}\n")

# --- Validierung gegen Gold-Labels der Handsuche (Zelle 05) ----------------------
df_gold = pd.read_csv(GOLD / "02_05_gold_stichprobe.csv")
print("=== Validierung gegen Handsuche (Gold) ===")
for _, g in df_gold[df_gold["gold_quelle"] == "Handsuche"].iterrows():
    z = df_rest[df_rest["pfad"] == g["pfad"]].iloc[0]
    code_ziel = (z["kandidaten"][0] if z["n_kandidaten"] == 1
                 else "fehlt" if z["n_kandidaten"] == 0 else "mehrere Kandidaten")
    print(f"  Nr. {int(g['nr'])}: Code = {z['kategorie']} -> {code_ziel!r}")
    print(f"          Gold = {g['gold_ziel']!r} | übereinstimmend: {code_ziel == g['gold_ziel']}")

# --- Grafik --------------------------------------------------------------------------
df_plot_k = df_rest["kategorie"].value_counts().sort_index(ascending=False)
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.barh(df_plot_k.index, df_plot_k.values, color=FARBEN["gold"])
for y, v in enumerate(df_plot_k.values):
    ax.text(v + 2, y, str(v), va="center", fontsize=12)
ax.set_xlabel("eindeutige Crate-Pfade ohne Treffer in Stufe 1–3")
ax.set_title(f"{len(df_rest)} Fehltreffer nach Namensabgleich im Index "
             "(Namensgleichheit ≠ Inhaltsgleichheit)")
figur_speichern(fig, "02_06_fehltreffer_kategorien", FIG)
plt.show()

### Erkenntnisse Zelle 06: Fehltreffer nach Namensabgleich

**Evidenz** (404 eindeutige Crate-Pfade ohne Treffer in Stufe 1–3; Vergleich über Kleinschreibung(NFC(Dateiname)) gegen 5.236 Audiodateien)

| Kategorie | Pfade | Anteil an 404 | Anteil an 2.754 |
|---|---|---|---|
| a Name 1x im Index, anderer Ordner | 166 | 41,1 % | 6,0 % |
| b Name mehrfach im Index | 0 | 0 % | 0 % |
| c Name erst nach Mojibake-Reparatur | 0 | 0 % | 0 % |
| d Name nicht im Index, Ordner vorhanden | 8 | 2,0 % | 0,3 % |
| e Name nicht im Index, Ordner nicht vorhanden | 230 | 56,9 % | 8,4 % |

- Zusatzmerkmal: 25 der 166 a-Pfade (15,1 %) haben einen gleichnamigen Ordner an anderer Stelle; in d und e: 0.
- NFD: alle 25 NFD-Fehltreffer liegen in e; keine Überschneidung mit den 25 Fällen „Ordnername anderswo“.
- Mojibake: 1 reparierbarer Audio-Dateiname im Index, kein Crate-Verweis darauf.
- Validierung gegen Handsuche: 3/3 übereinstimmend (Nr. 12 mit identischem Zielpfad, Nr. 11 und 13 ohne Kandidat).

**Aussagen**

- Über Namensgleichheit liegt für 2.516 von 2.754 Pfaden (91,4 %) ein Kandidat vor (Stufe 1: 2.075, Stufe 2: 275, Kategorie a: 166).
- Für 238 Pfade (8,6 %) liefert der Dateiname keinen Kandidaten.
- Mehrdeutigkeit (b) und Mojibake (c) treten bei den Fehltreffern nicht auf.

**Grenzen**

- Namensgleichheit belegt keine Inhaltsgleichheit; Kategorie a ist ein Kandidat, nicht „sicher“.
- Ursachen sind nicht geprüft; Hypothesen: a verschoben (25 Fälle mit gleichnamigem Ordner an anderer Stelle; Beispiel Gold-Stichprobe Nr. 12, laut Projektinhaber versehentlich verschoben); d/e gelöscht, umbenannt oder umbenannt und verschoben.

**Konsequenz**

- Inhaltsbeleg und Suche ohne Dateinamen erfordern Metadaten → Serato `database V2` (Zelle 07).
- Fall Gold-Stichprobe Nr. 12 = erster bestätigter Reparaturvorschlag für den Pflichtbericht (Ausführung durch den Nutzer, P1/P4).

In [ ]:
# =============================================================================
# Zelle 07 – Crates (bis 2016) vs. Subcrates (2021): Vergleich nach Inhalt
# =============================================================================
# Anlass (Notebook 01, Zelle 11): Subcrates 203 Dateien (2021), Crates 167
#   (2009–2016); 188 Subcrate-Namen mit "%%"; Namensabgleich nur 6 Treffer.
# Frage: Welcher Ordner ist die Wahrheitsquelle (H4: Subcrates = jüngerer Stand)?
# Methode:
#   1 beide Ordner mit parse_crate() lesen, Pfade NFC-normalisiert als Menge
#   2 Paare über den letzten Namensteil nach "%%" (Kleinschreibung, NFC);
#     je Paar Relation der Pfadmengen + Jaccard = |A∩B| / |A∪B|
#   3 Pfadebene: eindeutige Pfade nur Crates / nur Subcrates / beide,
#     je Gruppe Anteil mit Treffer im Index (nach NFC)
# Datenschutz: Namen und Paare privat (data/interim), public nur Zählwerte.
# Voraussetzung: Zellen 01–03 (parse_crate, unicodedata, idx_nfc).
# =============================================================================

SUBCRATES = RAW / "serato" / "Subcrates"


def nfc(s: str) -> str:
    return unicodedata.normalize("NFC", s)


# --- 1. Beide Ordner parsen ---------------------------------------------------
zeilen = []
for quelle_ordner, dateien in [("Crates", sorted((RAW / "crates").glob("*.crate"))),
                        ("Subcrates", sorted(SUBCRATES.glob("*.crate")))]:
    for f in dateien:
        r = parse_crate(f)
        zeilen.append({"ordner": quelle_ordner, "datei": f.stem,
                       "blatt": nfc(f.stem.split("%%")[-1]).lower(),
                       "ebenen": f.stem.count("%%") + 1,
                       "vollstaendig": r["vollstaendig"],
                       "n_eintraege": len(r["pfade"]),
                       "pfade": frozenset(nfc(p) for p in r["pfade"])})
df_alle_crates = pd.DataFrame(zeilen)

print("=== 1: Lesen ===")
print(df_alle_crates.groupby("ordner").agg(
    dateien=("datei", "size"), vollstaendig=("vollstaendig", "sum"),
    eintraege=("n_eintraege", "sum"), ebenen_max=("ebenen", "max")).to_string())

# --- 2. Paare über den letzten Namensteil -------------------------------------
def relation(a: frozenset, b: frozenset) -> str:
    """Relation der Pfadmenge Crate (a) zu Subcrate (b)."""
    if a == b:
        return "identisch"
    if a < b:
        return "Subcrate Obermenge"
    if a > b:
        return "Subcrate Teilmenge"
    return "Überlappung" if a & b else "disjunkt"


c = df_alle_crates[df_alle_crates["ordner"] == "Crates"]
s = df_alle_crates[df_alle_crates["ordner"] == "Subcrates"]

df_crate_paare = c.merge(s, on="blatt", suffixes=("_c", "_s"))
df_crate_paare["relation"] = [relation(a, b) for a, b in
                              zip(df_crate_paare["pfade_c"], df_crate_paare["pfade_s"])]
df_crate_paare["jaccard"] = [len(a & b) / len(a | b) if (a | b) else 1.0 for a, b in
                             zip(df_crate_paare["pfade_c"], df_crate_paare["pfade_s"])]
df_crate_paare.drop(columns=["pfade_c", "pfade_s"]).to_csv(
    PROJECT_ROOT / "data/interim/02_07_crate_paare.csv", index=False, encoding="utf-8")  # privat

# je Crate (Crates-Ordner) der Partner mit höchstem Jaccard
beste = (df_crate_paare.sort_values("jaccard", ascending=False)
         .drop_duplicates("datei_c"))
df_rel = c[["datei"]].merge(beste[["datei_c", "relation", "jaccard"]],
                            left_on="datei", right_on="datei_c", how="left")
df_rel["relation"] = df_rel["relation"].fillna("kein Namenspartner")

df_relation_zaehlung = (df_rel.groupby("relation", as_index=False)
                        .agg(crates=("datei", "size"), jaccard_median=("jaccard", "median")))
n_partner = df_crate_paare.groupby("datei_c").size()
print("\n=== 2: Crates-Ordner, Relation zum besten Subcrate-Partner ===")
print(df_relation_zaehlung.to_string(index=False))
print(f"Crates mit >1 Namenspartner: {(n_partner > 1).sum()}")
print(f"Subcrates ohne Namenspartner in Crates: {(~s['blatt'].isin(c['blatt'])).sum()} von {len(s)}")
tabelle_speichern(df_relation_zaehlung, "02_07_crates_subcrates_relation")

# --- 3. Pfadebene ---------------------------------------------------------------
P_c = frozenset().union(*c["pfade"])
P_s = frozenset().union(*s["pfade"])
gruppen = {"nur Crates": P_c - P_s, "nur Subcrates": P_s - P_c, "beide": P_c & P_s}
df_pfadebene = pd.DataFrame([
    {"gruppe": g, "pfade": len(m), "im_index_nach_nfc": sum(p in idx_nfc for p in m)}
    for g, m in gruppen.items()])
df_pfadebene["anteil_im_index"] = (df_pfadebene["im_index_nach_nfc"]
                                   / df_pfadebene["pfade"]).round(3)
print("\n=== 3: eindeutige Pfade (NFC) ===")
print(df_pfadebene.to_string(index=False))
print(f"Summe: {df_pfadebene['pfade'].sum()} | Crates gesamt {len(P_c)} | Subcrates gesamt {len(P_s)}")
tabelle_speichern(df_pfadebene, "02_07_pfadebene_crates_subcrates")

# --- Grafik ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(df_relation_zaehlung["relation"], df_relation_zaehlung["crates"], color=FARBEN["gold"])
ax.set_xlabel("Anzahl Crates (Ordner Crates, n = 167)")
ax.set_title("Crates vs. Subcrates: Relation der Pfadmengen")
figur_speichern(fig, "02_07_crates_subcrates_relation", FIG)
plt.show()

### Befund und Entscheidung Zelle 07 – Crates vs. Subcrates (Entscheidung 09.11.2026)

**Befund (berechnet, Pfade NFC-normalisiert):**
- Beide Ordner vollständig gelesen: Crates 167/167 (Stand 2009–2016), Subcrates 203/203 (Stand 07–11/2021, 188 mit Hierarchie `%%`).
- Je alter Crate, Vergleich mit der Subcrate gleichen Namens (letzter Namensteil):
  identisch 97 | Subcrate Teilmenge 8 | Subcrate Obermenge 3 | Überlappung 3 | kein Namenspartner 56 (Summe 167).
- 91 von 203 Subcrates ohne Namenspartner im alten Ordner.
- Eindeutige Pfade: nur alt 179 (20,7 % auf dem Stick) | nur Subcrates 765 (88,5 %) | beide 2.380 (89,7 %).
- Nach NFC-Normalisierung 2.559 statt 2.754 Pfade im alten Ordner: 195 Pfade lagen in NFC- und NFD-Form vor.

**Hypothese (nicht belegt):** Subcrates sind ein jüngerer, gepflegter Stand. Die Herkunft ist unbekannt.

**Entscheidung d:** Der alte Ordner `Crates` bleibt die Wahrheitsquelle; die Zahlen aus den Zellen 02–06 bleiben gültig.
Subcrates sind eine zusätzliche Quelle mit niedrigerer Priorität:
- identisch → keine Aktion;
- abweichend → Variante `<name>_2021` und Konflikt im Bericht;
- ohne Partner → eigene Playlist `<name>_2021`, im Bericht als „neue Quelle".

Der Nutzer entscheidet. Begründung: Bei unbekannter Herkunft wird keine Quelle ungeprüft bevorzugt; nichts geht verloren (P4, Human in the Loop).

In [ ]:
# =============================================================================
# Zelle 08 – Serato "database V2": Format, Felder, Abdeckung der Crate-Pfade
# =============================================================================
# Anlass: Zelle 06 – 238 Crate-Pfade ohne Namenskandidaten im Index; Metadaten
#   (Titel, Interpret, Dauer, Größe) könnten eine Suche ohne Dateinamen erlauben.
# Hypothese Format (Zelle 10, Notebook 01: beginnt mit 'vrsn'): TLV wie Crates,
#   Track = 'otrk', darin Felder. Dekodierung nach erstem Buchstaben des Tags
#   (t/p = Text UTF-16-BE, u = 4-Byte-Zahl, s = 2-Byte-Zahl, b = 1-Byte-Flag)
#   ist eine Hypothese; Prüfung: Datei und jeder Track gehen exakt auf.
# Messung: 1 Integrität | 2 Feld-Befüllung | 3 Abdeckung alter Crates,
#   Subcrates und der Stufe-4-Pfade (Zelle 03) über Pfad 'pfil' (NFC).
# Datenschutz: Track-Tabelle privat (data/interim), public nur Zählwerte.
# Voraussetzung: Zellen 01–07 (df_pfade, P_c, P_s, nfc).
# =============================================================================

from collections import Counter

DB = RAW / "serato" / "database V2"


def tlv_bloecke(b: bytes) -> tuple[list, bool]:
    """Zerlegt Bytes in (Tag, Wert)-Blöcke; zweiter Wert: geht exakt auf?"""
    i, bloecke = 0, []
    while i + 8 <= len(b):
        tag, n = b[i:i + 4], int.from_bytes(b[i + 4:i + 8], "big")
        bloecke.append((tag.decode("latin-1"), b[i + 8:i + 8 + n]))
        i += 8 + n
    return bloecke, i == len(b)


def feldwert(tag: str, wert: bytes):
    """Dekodierung nach Tag-Präfix; unbekannt -> nur Länge (keine Annahme)."""
    try:
        if tag[0] in "tp":
            return wert.decode("utf-16-be")
        if tag[0] == "u" and len(wert) == 4:
            return int.from_bytes(wert, "big")
        if tag[0] == "s" and len(wert) == 2:
            return int.from_bytes(wert, "big")
        if tag[0] == "b" and len(wert) == 1:
            return wert[0]
    except UnicodeDecodeError:
        pass
    return f"<{len(wert)} Byte>"


def pfad_schluessel(p) -> str | None:
    """Vergleichsform: NFC, ohne führenden '/'."""
    return nfc(p).lstrip("/") if isinstance(p, str) else None


def befuellt(v) -> bool:
    return pd.notna(v) and str(v).strip() != ""


# --- 1. Integrität -------------------------------------------------------------
bloecke, datei_ok = tlv_bloecke(DB.read_bytes())
tracks, tracks_ok = [], 0
for tag, wert in bloecke:
    if tag == "otrk":
        felder, ok = tlv_bloecke(wert)
        tracks_ok += ok
        tracks.append({t: feldwert(t, w) for t, w in felder})
df_db = pd.DataFrame(tracks)
df_db["schluessel"] = df_db["pfil"].map(pfad_schluessel)
df_db.to_csv(PROJECT_ROOT / "data/interim/02_08_database_v2.csv",
             index=False, encoding="utf-8")                              # privat

print("=== 1: Integrität ===")
print(f"Datei geht exakt auf      : {datei_ok}")
print(f"Blöcke oberste Ebene      : {dict(Counter(t for t, _ in bloecke))}")
print(f"Tracks (otrk)             : {len(df_db)} | davon exakt aufgehend: {tracks_ok}")
print(f"pfil mit führendem '/'    : {df_db['pfil'].dropna().str.startswith('/').sum()}")
print(f"doppelte Pfade (Schlüssel): {df_db['schluessel'].duplicated().sum()}")

# --- 2. Feld-Befüllung -------------------------------------------------------------
df_db_felder = pd.DataFrame({
    "feld": [c for c in df_db.columns if c != "schluessel"],
})
df_db_felder["befuellt"] = df_db_felder["feld"].map(lambda c: df_db[c].map(befuellt).sum())
df_db_felder["anteil"] = (df_db_felder["befuellt"] / len(df_db)).round(3)
df_db_felder["typ"] = df_db_felder["feld"].map(
    lambda c: type(df_db[c].dropna().iloc[0]).__name__ if df_db[c].notna().any() else "-")
df_db_felder = df_db_felder.sort_values("anteil", ascending=False)
print("\n=== 2: Felder ===")
print(df_db_felder.to_string(index=False))
tabelle_speichern(df_db_felder, "02_08_database_felder")

# --- 3. Abdeckung über den Pfad -------------------------------------------------------
db_idx = df_db.dropna(subset=["schluessel"]).drop_duplicates("schluessel").set_index("schluessel")
gruppen = {
    "alte Crates": {pfad_schluessel(p) for p in P_c},
    "Subcrates": {pfad_schluessel(p) for p in P_s},
    "alte Crates, Stufe 4": {pfad_schluessel(p) for p in
                             df_pfade.loc[df_pfade["treffer"] == "4 kein Treffer", "pfad"]},
}
zeilen = []
for g, menge in gruppen.items():
    in_db = [k for k in menge if k in db_idx.index]
    zeile = {"gruppe": g, "pfade": len(menge), "in_database": len(in_db)}
    for feld in ["tsng", "tart", "tlen", "tsiz"]:
        zeile[f"mit_{feld}"] = (sum(befuellt(db_idx.at[k, feld]) for k in in_db)
                                if feld in db_idx.columns else 0)
    zeilen.append(zeile)
df_db_abdeckung = pd.DataFrame(zeilen)
df_db_abdeckung["anteil_in_database"] = (df_db_abdeckung["in_database"]
                                         / df_db_abdeckung["pfade"]).round(3)
print("\n=== 3: Abdeckung (Pfad-Schlüssel NFC) ===")
print(df_db_abdeckung.to_string(index=False))
tabelle_speichern(df_db_abdeckung, "02_08_database_abdeckung")

# --- Grafik ------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(df_db_abdeckung["gruppe"], df_db_abdeckung["anteil_in_database"], color=FARBEN["blau"])
ax.set_xlim(0, 1)
ax.set_xlabel("Anteil der Pfade mit Eintrag in database V2")
ax.set_title("Abdeckung der Crate-Pfade durch database V2")
figur_speichern(fig, "02_08_database_abdeckung", FIG)
plt.show()

### Befund Zelle 08 – Serato `database V2`

**Format:** TLV wie die Crates; die Datei und alle 5.478 Track-Blöcke (`otrk`) gehen exakt auf. Die Dekodierung nach Tag-Präfix (t/p Text UTF-16-BE, u/s/b Zahl) war für alle Felder erfolgreich.

**Felder für die Suche ohne Dateinamen:** `tsng` 99,9 % | `tart` 89,1 % | `tlen` 83,3 % | `tsiz` 66,1 % befüllt.
Die Bedeutung der Felder ist aus den Namen abgeleitet (Hypothese) und wird mit den Tags der Dateien geprüft (Zelle 11).

**Abdeckung über den Pfad (NFC):** alte Crates 2.477/2.559 (96,8 %) | Subcrates 3.145/3.145 (100 %) | Stufe-4-Pfade 308/386 (79,8 %), davon 304 mit Dauer.

**Folge:** Für 308 Pfade ohne Index-Treffer liegen Titel und meist Dauer vor; das ist die Grundlage für eine inhaltliche Suche (Stufe 4.3, Inhaltsbelege). Für 78 Pfade gibt es keine Metadaten.

**Offen:** 746 doppelte Pfad-Schlüssel (Hypothese: NFC/NFD-Doppel); Überschneidung mit den 238 Pfaden ohne Namenskandidaten (Zelle 06).

In [ ]:
# =============================================================================
# Zelle 09 – database V2: doppelte Pfade und Metadaten je Fehltreffer-Kategorie
# =============================================================================
# Offen aus Zelle 08:
#   A 746 doppelte Pfad-Schlüssel. Hypothese: derselbe Pfad in NFC- und NFD-Form.
#     Prüfung je Gruppe: unterscheiden sich die Rohpfade (nur Normalform) oder
#     sind sie Zeichen-identisch (echte Doppeleinträge)?
#   B Wie viele Fehltreffer je Kategorie (Zelle 06: a 166, d 8, e 230) haben
#     Metadaten in database V2? Für d/e (238, kein Namenskandidat) entscheidet
#     das, ob eine inhaltliche Suche überhaupt möglich ist.
# Zählung B je Rohpfad (404 Zeilen in df_rest), Zuordnung über NFC-Schlüssel.
# Datenschutz: public nur Zählwerte.
# Voraussetzung: Zellen 01–08 (df_db, db_idx, df_rest, pfad_schluessel, befuellt).
# =============================================================================

# --- A: doppelte Pfad-Schlüssel ------------------------------------------------
df_dup = df_db[df_db["schluessel"].duplicated(keep=False)]
df_dup_gruppen = (df_dup.groupby("schluessel")["pfil"]
                  .agg(zeilen="size", rohformen="nunique").reset_index())
df_dup_gruppen["art"] = df_dup_gruppen["rohformen"].map(
    lambda n: "Rohpfade verschieden (Normalform)" if n > 1 else "Rohpfade identisch")
df_dup_zaehlung = (df_dup_gruppen.groupby("art", as_index=False)
                   .agg(gruppen=("schluessel", "size"), zeilen=("zeilen", "sum")))
df_dup_zaehlung["ueberzaehlig"] = df_dup_zaehlung["zeilen"] - df_dup_zaehlung["gruppen"]
print("=== A: doppelte Pfad-Schlüssel in database V2 ===")
print(df_dup_zaehlung.to_string(index=False))
print(f"Summe überzählig: {df_dup_zaehlung['ueberzaehlig'].sum()} (Zelle 08: 746)")
tabelle_speichern(df_dup_zaehlung, "02_09_database_duplikate")

# --- B: Metadaten je Fehltreffer-Kategorie ---------------------------------------
df_rest["db_schluessel"] = df_rest["pfad"].map(pfad_schluessel)
df_rest["in_database"] = df_rest["db_schluessel"].isin(db_idx.index)
for feld in ["tsng", "tart", "tlen", "tsiz"]:
    df_rest[f"db_{feld}"] = [befuellt(db_idx.at[k, feld]) if ok else False
                             for k, ok in zip(df_rest["db_schluessel"], df_rest["in_database"])]

df_kat_db = (df_rest.groupby("kategorie", as_index=False)
             .agg(pfade=("pfad", "size"), in_database=("in_database", "sum"),
                  mit_tsng=("db_tsng", "sum"), mit_tart=("db_tart", "sum"),
                  mit_tlen=("db_tlen", "sum"), mit_tsiz=("db_tsiz", "sum")))
df_kat_db["anteil_in_database"] = (df_kat_db["in_database"] / df_kat_db["pfade"]).round(3)
print("\n=== B: Fehltreffer je Kategorie mit Metadaten in database V2 ===")
print(df_kat_db.to_string(index=False))
print(f"Summe Pfade: {df_kat_db['pfade'].sum()} (Zelle 06: 404)")
tabelle_speichern(df_kat_db, "02_09_fehltreffer_kategorie_database")

### Befund Zelle 09 – Doppelte Pfade und Metadaten je Fehltreffer-Kategorie

**A – doppelte Pfade:** Alle 746 Gruppen in `database V2` bestehen aus genau 2 Einträgen, deren Rohpfade sich nur in der Normalform (NFC/NFD) unterscheiden. Es gibt keine Zeichen-identischen Doppeleinträge. Folge für den Parser: Einträge vor dem Abgleich nach NFC zusammenführen.

**B – Metadaten je Kategorie (404 Fehltreffer, Zelle 06):**
a 88/166 in database (53 %) | d 8/8 (100 %) | e 230/230 (100 %).
Alle 238 Pfade ohne Namenskandidaten haben Metadaten (Titel 238, Dauer 235). Die 78 Fehltreffer ohne Metadaten liegen alle in Kategorie a und haben einen Namenskandidaten.
**Damit hat jeder Fehltreffer mindestens eine Belegquelle:** einen Namenskandidaten oder Metadaten.

**Hypothesen (nicht geprüft):** d/e-Dateien existierten 2021 noch in der Serato-Bibliothek; a-Pfade wurden vor 2021 verschoben, und die Datenbank führt eventuell den neuen Ort.

**Folge:** Die inhaltliche Suche (Titel, Interpret und Dauer gegen die Datei-Tags, mutagen) ist für alle 238 d/e-Pfade möglich. Ob dabei die richtige Datei gefunden wird, zeigt erst der Abgleich mit den Tags (Zelle 11) und das Gold-Set.

In [ ]:
# =============================================================================
# Zelle 10 – Playlist-Ebene: Wiederherstellbarkeit je Crate mit Bootstrap-KI
# =============================================================================
# Zweck: Nutzersicht – wie viele Crates sind vollständig wiederherstellbar?
#   Bisher Song-Ebene (eindeutige Pfade, Zellen 03–09).
# Belegklasse je Rohpfad (erste zutreffende, Rechenvorschrift):
#   1 sicher          Treffer Stufe 1 oder 2 (exakt / nach NFC), Zelle 03
#   2 Namenskandidat  Kategorie a (Name 1x im Index, anderer Ordner), Zelle 06
#   3 nur Metadaten   Kategorie d/e mit Eintrag in database V2, Zelle 09
#   4 ohne Beleg      sonst (Zelle 09 erwartet 0)
# Klassen 2/3 sind KEINE Treffer, nur Kandidaten bzw. Suchgrundlage (P5).
# Je Crate (alte Crates, n = 167): Anteile je Klasse über eindeutige Pfade.
# Unsicherheit: Bootstrap über Crates (10.000 Wiederholungen, SEED) für den
#   mittleren Anteil "sicher" je Crate -> 95-%-Perzentil-Intervall.
# Datenschutz: Crate-Namen nur privat (data/interim), public nur Zählwerte.
# Voraussetzung: Zellen 01–09 (df_eintraege, df_pfade, df_rest, SEED).
# =============================================================================

import numpy as np

KL_SICHER, KL_NAME, KL_META, KL_OHNE = ("1 sicher", "2 Namenskandidat",
                                        "3 nur Metadaten", "4 ohne Beleg")

# --- Belegklasse je Rohpfad -------------------------------------------------------
belegklasse = {p: KL_SICHER for p in
               df_pfade.loc[df_pfade["treffer"].isin(["1 exakt", "2 erst nach NFC"]), "pfad"]}
for _, z in df_rest.iterrows():
    if z["kategorie"] == KAT_A:
        belegklasse[z["pfad"]] = KL_NAME
    elif z["kategorie"] in (KAT_D, KAT_E) and z["in_database"]:
        belegklasse[z["pfad"]] = KL_META
    else:
        belegklasse[z["pfad"]] = KL_OHNE

# --- Je Crate: Anteile über eindeutige Pfade ----------------------------------------
df_crate_pfade = df_eintraege[["crate", "pfad"]].drop_duplicates()
df_crate_pfade["klasse_beleg"] = df_crate_pfade["pfad"].map(belegklasse)
assert df_crate_pfade["klasse_beleg"].notna().all(), "Pfad ohne Belegklasse"

df_je_crate = (pd.crosstab(df_crate_pfade["crate"], df_crate_pfade["klasse_beleg"],
                           normalize="index")
               .reindex(columns=[KL_SICHER, KL_NAME, KL_META, KL_OHNE], fill_value=0.0))
df_je_crate["n_pfade"] = df_crate_pfade.groupby("crate").size()
df_je_crate.to_csv(PROJECT_ROOT / "data/interim/02_10_wiederherstellbarkeit_je_crate.csv",
                   encoding="utf-8")                                     # privat

# --- Kennzahlen ---------------------------------------------------------------------
anteil_sicher = df_je_crate[KL_SICHER].to_numpy()
rng = np.random.default_rng(SEED)
boot = np.array([rng.choice(anteil_sicher, size=len(anteil_sicher), replace=True).mean()
                 for _ in range(10_000)])
ki_unten, ki_oben = np.percentile(boot, [2.5, 97.5])

df_playlist_kennzahlen = pd.DataFrame([
    {"kennzahl": "Crates gesamt", "wert": len(df_je_crate)},
    {"kennzahl": "Crates 100 % sicher (Stufe 1-2)", "wert": int((anteil_sicher == 1).sum())},
    {"kennzahl": "Crates >= 90 % sicher", "wert": int((anteil_sicher >= 0.9).sum())},
    {"kennzahl": "Crates < 50 % sicher", "wert": int((anteil_sicher < 0.5).sum())},
    {"kennzahl": "Crates mit Pfad ohne Beleg", "wert": int((df_je_crate[KL_OHNE] > 0).sum())},
    {"kennzahl": "Mittel Anteil sicher je Crate", "wert": round(anteil_sicher.mean(), 3)},
    {"kennzahl": "Bootstrap 95-%-KI unten", "wert": round(ki_unten, 3)},
    {"kennzahl": "Bootstrap 95-%-KI oben", "wert": round(ki_oben, 3)},
    {"kennzahl": "Median Anteil sicher je Crate", "wert": round(float(np.median(anteil_sicher)), 3)},
])
print("=== Playlist-Ebene (alte Crates) ===")
print(df_playlist_kennzahlen.to_string(index=False))
tabelle_speichern(df_playlist_kennzahlen, "02_10_playlist_ebene_kennzahlen")

# --- Grafik: Verteilung Anteil sicher je Crate --------------------------------------
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(anteil_sicher, bins=np.linspace(0, 1, 11), color=FARBEN["gold"], edgecolor=FARBEN["hintergrund"])
ax.axvline(anteil_sicher.mean(), color=FARBEN["blau"], linestyle="--",
           label=f"Mittel {anteil_sicher.mean():.3f} (95-%-KI {ki_unten:.3f}–{ki_oben:.3f})")
ax.set_xlabel("Anteil sicher auflösbarer Pfade je Crate (Stufe 1–2)")
ax.set_ylabel("Anzahl Crates")
ax.set_title(f"Wiederherstellbarkeit je Crate (n = {len(df_je_crate)})")
ax.legend()
figur_speichern(fig, "02_10_wiederherstellbarkeit_je_crate", FIG)
plt.show()

### Befund Zelle 10 – Playlist-Ebene (alte Crates)

**Ergebnis (Stufe 1–2 = sicher):** 125 von 167 Crates (74,9 %) sind vollständig sicher wiederherstellbar; 145 (86,8 %) zu mindestens 90 %; 7 (4,2 %) zu unter 50 %.
Mittlerer Anteil je Crate 93,2 % (Bootstrap 95-%-KI 90,1–96,0 %, 10.000 Wiederholungen), Median 100 %.
Jeder Pfad hat mindestens eine Belegquelle (0 Crates mit Pfad ohne Beleg).

**Einordnung:** Auf Pfad-Ebene sind 85,3 % sicher (2.350/2.754). Der höhere Wert je Crate entsteht durch die Gewichtung (jede Crate zählt gleich). Hypothese: Die Fehltreffer konzentrieren sich in wenigen großen Crates.

**Grenze:** Der Bootstrap nimmt unabhängige Crates an; die Crates teilen sich aber Pfade (im Mittel 2,9 Crates je Pfad), daher ist das Intervall vermutlich zu schmal.

**Folge für das Projektziel:** Die deterministische Baseline (Stufen 1–2) stellt drei Viertel der Playlists ohne Nutzereingriff vollständig wieder her. Für das restliche Viertel sind Namenskandidaten und Metadaten die Grundlage der Stufen 4.3/4.4.

In [ ]:
# =============================================================================
# Zelle 11 – Audio-Tags (tinytag): Probelauf mit Laufzeit-Hochrechnung
# =============================================================================
# Zweck: Vor dem Lesen aller 5.236 Audiodateien vom Stick messen, wie lange es
#   dauert und wie vollständig die Tags sind (Grundsatz: erst messen).
# Zugriff: nur lesend auf den Stick (P7-Ausnahme, Entscheidung 07.11.2026;
#   Bibliothek tinytag, Entscheidung 11.11.2026: MIT, kann nicht schreiben -> P1).
# Stichprobe: je Endung bis zu 40 Dateien (SEED), geschichtet, weil wav/aif
#   deutlich größer sind als mp3 und die Laufzeit anders sein kann.
# Hochrechnung: mittlere Zeit je Datei der Endung x Anzahl Dateien der Endung.
# Datenschutz: Einzelwerte privat (data/interim), public nur Zählwerte/Zeiten.
# Voraussetzung: Zellen 01–06 (df_audio, STICK_LW, SEED); Stick eingesteckt.
# =============================================================================

import time
from tinytag import TinyTag

STICK_WURZEL = Path(STICK_LW + "/")
assert (STICK_WURZEL / "Musik").is_dir(), f"Stick nicht unter {STICK_LW} gefunden – einstecken."

FELDER = ["title", "artist", "album", "track", "year", "duration"]


def tags_lesen(rel: str) -> dict:
    """Liest Tags und Dauer einer Datei; misst die Zeit; Fehler werden erfasst, nicht verschluckt."""
    t0 = time.perf_counter()
    try:
        tag = TinyTag.get(STICK_WURZEL / rel, image=False)
        werte = {f: getattr(tag, f, None) for f in FELDER}
        fehler = None
    except Exception as e:                       # einzelne Datei darf nicht alles stoppen
        werte, fehler = {f: None for f in FELDER}, repr(e)
    return {"pfad_relativ": rel, **werte, "fehler": fehler,
            "sekunden": time.perf_counter() - t0}


# --- Geschichtete Stichprobe -----------------------------------------------------
# GroupBy.sample behält alle Spalten (apply entfernt in pandas 3 die Gruppierungsspalte).
# n = 40 je Endung zulässig: kleinste Gruppe aif hat 46 Dateien.
df_probe = df_audio.groupby("endung").sample(n=40, random_state=SEED)
df_probe_tags = pd.DataFrame([tags_lesen(p) for p in df_probe["pfad_relativ"]])
df_probe_tags["endung"] = df_probe["endung"].to_numpy()
df_probe_tags.to_csv(PROJECT_ROOT / "data/interim/02_11_tinytag_probe.csv",
                     index=False, encoding="utf-8")                      # privat

# --- Auswertung je Endung ----------------------------------------------------------
n_gesamt = df_audio["endung"].value_counts()
df_tinytag_probe = (df_probe_tags.groupby("endung", as_index=False)
    .agg(n_probe=("pfad_relativ", "size"),
         fehler=("fehler", lambda s: s.notna().sum()),
         sek_median=("sekunden", "median"), sek_mittel=("sekunden", "mean"),
         sek_max=("sekunden", "max"),
         mit_title=("title", lambda s: s.map(befuellt).sum()),
         mit_artist=("artist", lambda s: s.map(befuellt).sum()),
         mit_duration=("duration", lambda s: s.map(befuellt).sum())))
df_tinytag_probe["n_gesamt"] = df_tinytag_probe["endung"].map(n_gesamt)
df_tinytag_probe["hochrechnung_min"] = (df_tinytag_probe["sek_mittel"]
                                        * df_tinytag_probe["n_gesamt"] / 60).round(1)
print(df_tinytag_probe.round(3).to_string(index=False))
print(f"\nHochrechnung gesamt: {df_tinytag_probe['hochrechnung_min'].sum():.1f} min "
      f"für {df_tinytag_probe['n_gesamt'].sum()} Dateien")
tabelle_speichern(df_tinytag_probe.round(3), "02_11_tinytag_probelauf")

In [ ]:
# =============================================================================
# Zelle 12 – Audio-Tags aller Audiodateien (tinytag, nur lesend vom Stick)
# =============================================================================
# Begründung Vollauf: Probelauf Zelle 11 – 0 Fehler in 120 Dateien,
#   Hochrechnung 4,4 min (vermutlich zu optimistisch: Dateien im Cache).
# Ergebnis privat (data/interim) als Grundlage für V2-Abgleich mit database V2.
# Public: Befüllung je Endung, gemessene Gesamtlaufzeit.
# Voraussetzung: Zellen 01–11 (df_audio, tags_lesen, befuellt).
# =============================================================================

t_start = time.perf_counter()
zeilen = []
for i, p in enumerate(df_audio["pfad_relativ"], start=1):
    zeilen.append(tags_lesen(p))
    if i % 500 == 0:
        print(f"  {i:>5} / {len(df_audio)}  ({(time.perf_counter() - t_start) / 60:.1f} min)")
laufzeit_min = (time.perf_counter() - t_start) / 60

df_tags = pd.DataFrame(zeilen)
df_tags["endung"] = df_audio["endung"].to_numpy()
df_tags.to_csv(PROJECT_ROOT / "data/interim/02_12_audio_tags.csv",
               index=False, encoding="utf-8")                            # privat

df_tags_befuellung = (df_tags.groupby("endung", as_index=False)
    .agg(dateien=("pfad_relativ", "size"),
         fehler=("fehler", lambda s: s.notna().sum()),
         **{f"mit_{f}": (f, lambda s: s.map(befuellt).sum()) for f in FELDER}))
print(f"\nLaufzeit gesamt: {laufzeit_min:.1f} min für {len(df_tags)} Dateien")
print(df_tags_befuellung.to_string(index=False))
_ = tabelle_speichern(df_tags_befuellung, "02_12_audio_tags_befuellung")

### Befund Zellen 11–12 – Audio-Tags (tinytag, nur lesend)

**Messung vor dem Vollauf (Zelle 11):** Probelauf mit 120 Dateien (40 je Endung): 0 Fehler, Hochrechnung 4,4 min. Gemessen im Vollauf (Zelle 12): 4,3 min für 5.236 Dateien, 0 Fehler (Rule of Three: Fehlerrate < 0,06 % bei 95 % Konfidenz).

**Befüllung (gesamt / mp3 / wav / aif):**
Dauer 100 % in allen Formaten | Titel 93,6 / 93,6 / 95,1 / 89,1 % | Interpret 86,6 / 92,1 / 0,4 / 0 % | Tracknummer 61,3 / 65,2 / 0 / 0 %.

**Folgen:**
- Die Dauer ist das einzige für alle Dateien verfügbare Inhaltsmerkmal → Hauptmerkmal für den Abgleich mit `database V2` (V2) und für die Suche in Kategorie d/e.
- Für wav/aif gibt es keinen Interpreten; der Abgleich stützt sich dort auf Titel und Dauer.
- `create_from_folder`: Die Tracknummer fehlt bei 34,8 % der mp3 und allen wav/aif → Rückfallregel für die Reihenfolge nötig (Entscheidung Phase 3).

**Technischer Befund pandas 3:** `groupby().apply()` übergibt die Gruppierungsspalte nicht mehr; geschichtete Stichprobe daher mit `groupby().sample()`.

In [ ]:
# =============================================================================
# Zelle 13 – Inhaltsbeleg Dauer: Kalibrierung an bekannt gleichen Dateien
# =============================================================================
# Frage: Wie stark weichen Serato-Dauer (database V2, 'tlen') und Datei-Dauer
#   (tinytag, Zelle 12) voneinander ab, wenn es nachweislich DIESELBE Datei ist?
# Referenz ("bekannt gleich"): Crate-Pfade mit Treffer Stufe 1/2 (Pfad = Datei).
# Prüfgruppe: Kategorie a (Name 1x anderswo, Zelle 06) mit database-Eintrag:
#   Serato-Dauer des alten Pfads vs. Datei-Dauer des Namenskandidaten.
# Ergebnis: Verteilung |Δ| in Sekunden je Gruppe, Anteile unter Schwellen.
#   Eine Toleranz wird hier NICHT festgelegt (Entscheidung nach Sichtung).
# Format 'tlen' (Vorbefund): "mm:ss.hh" -> Sekunden; nicht lesbar -> NaN, gezählt.
# Datenschutz: Paare privat (data/interim), public nur Kennzahlen.
# Voraussetzung: Zellen 01–12 (df_pfade, df_rest, db_idx, df_tags, pfad_schluessel).
# =============================================================================


def tlen_sekunden(v) -> float:
    """'09:17.37' -> 557.37; auch 'h:mm:ss.hh'. Nicht lesbar -> NaN."""
    if not isinstance(v, str) or ":" not in v:
        return np.nan
    try:
        teile = [float(x) for x in v.split(":")]
    except ValueError:
        return np.nan
    sek = 0.0
    for t in teile:
        sek = sek * 60 + t
    return sek


tag_dauer = dict(zip(df_tags["pfad_relativ"].map(nfc), df_tags["duration"]))


def db_dauer(crate_pfad: str) -> float:
    k = pfad_schluessel(crate_pfad)
    return tlen_sekunden(db_idx.at[k, "tlen"]) if k in db_idx.index else np.nan


# --- Referenz: Stufe 1/2 (Pfad = Datei) -----------------------------------------
ref = df_pfade[df_pfade["treffer"].isin(["1 exakt", "2 erst nach NFC"])]
df_ref = pd.DataFrame({
    "gruppe": "Referenz Stufe 1/2",
    "db_sek": ref["pfad"].map(db_dauer).to_numpy(),
    "datei_sek": ref["pfad"].map(lambda p: tag_dauer.get(nfc(p), np.nan)).to_numpy(),
})

# --- Prüfgruppe: Kategorie a mit database-Eintrag --------------------------------
a = df_rest[(df_rest["kategorie"] == KAT_A) & df_rest["in_database"]]
df_a = pd.DataFrame({
    "gruppe": "Kategorie a (Kandidat)",
    "db_sek": a["pfad"].map(db_dauer).to_numpy(),
    "datei_sek": a["kandidaten"].map(lambda k: tag_dauer.get(nfc(k[0]), np.nan)).to_numpy(),
})

df_dauer = pd.concat([df_ref, df_a], ignore_index=True)
df_dauer["abs_diff"] = (df_dauer["db_sek"] - df_dauer["datei_sek"]).abs()
df_dauer.to_csv(PROJECT_ROOT / "data/interim/02_13_dauer_paare.csv",
                index=False, encoding="utf-8")                           # privat

# --- Kennzahlen je Gruppe ----------------------------------------------------------
SCHWELLEN = [0.05, 0.1, 0.5, 1.0, 2.0]
zeilen = []
for g, d in df_dauer.groupby("gruppe"):
    x = d["abs_diff"].dropna()
    zeile = {"gruppe": g, "paare": len(d), "vergleichbar": len(x),
             "median": x.median(), "p95": x.quantile(0.95), "p99": x.quantile(0.99),
             "max": x.max()}
    zeile.update({f"anteil_<={s}s": round((x <= s).mean(), 3) for s in SCHWELLEN})
    zeilen.append(zeile)
df_dauer_kennzahlen = pd.DataFrame(zeilen).round(3)
print(df_dauer_kennzahlen.T.to_string())
_ = tabelle_speichern(df_dauer_kennzahlen, "02_13_dauer_differenz")

# --- Grafik: empirische Verteilungsfunktion |Δ| ------------------------------------
fig, ax = plt.subplots(figsize=(9, 4))
for (g, d), farbe in zip(df_dauer.groupby("gruppe"), [FARBEN["gold"], FARBEN["blau"]]):
    x = np.sort(d["abs_diff"].dropna().to_numpy())
    ax.step(np.maximum(x, 1e-3), np.arange(1, len(x) + 1) / len(x), where="post",
            color=farbe, label=f"{g} (n = {len(x)})")
ax.set_xscale("log")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))   # 0.001 statt 10⁻³ (Schrift ohne Hochzeichen)
ax.set_xlabel("|Dauer Serato − Dauer Datei| in s (log; 0 als 0,001 dargestellt)")
ax.set_ylabel("Anteil ≤ x")
ax.set_title("Inhaltsbeleg Dauer: bekannt gleiche Dateien vs. Namenskandidaten")
ax.legend()
figur_speichern(fig, "02_13_dauer_differenz", FIG)
plt.show()

In [ ]:
# =============================================================================
# Zelle 14 – Ausreißer Dauer-Abweichung > 2 s bei bekannt gleichen Dateien
# =============================================================================
# Befund Zelle 13: 3,2 % der Referenzpaare (Pfad = Datei) weichen > 2 s ab,
#   Maximum 8.548 s -> mindestens eine Dauer-Angabe ist dort falsch.
# Hypothese (ungeprüft): betrifft vor allem mp3 (Dauer bei variabler Bitrate
#   geschätzt). Prüfung: Ausreißer je Endung, Richtung (welche Seite größer),
#   Verhältnis Serato/Datei (Faktor nahe 2 o. Ä. deutet auf Systematik).
# Datenschutz: Pfade nur privat (reports/errors), public nur Zählwerte.
# Voraussetzung: Zellen 01–13.
# =============================================================================

endung_von = dict(zip(df_tags["pfad_relativ"].map(nfc), df_tags["endung"]))

df_ref_aus = pd.DataFrame({
    "pfad": ref["pfad"].to_numpy(),
    "db_sek": df_ref["db_sek"].to_numpy(),
    "datei_sek": df_ref["datei_sek"].to_numpy(),
})
df_ref_aus["endung"] = df_ref_aus["pfad"].map(lambda p: endung_von.get(nfc(p)))
df_ref_aus["abs_diff"] = (df_ref_aus["db_sek"] - df_ref_aus["datei_sek"]).abs()
df_ref_aus = df_ref_aus.dropna(subset=["abs_diff"])
df_ref_aus["ausreisser"] = df_ref_aus["abs_diff"] > 2.0
df_ref_aus["richtung"] = np.where(df_ref_aus["db_sek"] > df_ref_aus["datei_sek"],
                                  "Serato länger", "Datei länger")
df_ref_aus["verhaeltnis"] = df_ref_aus["db_sek"] / df_ref_aus["datei_sek"]

# --- Je Endung -------------------------------------------------------------------
df_aus_endung = (df_ref_aus.groupby("endung", as_index=False)
                 .agg(paare=("pfad", "size"), ausreisser=("ausreisser", "sum")))
df_aus_endung["anteil"] = (df_aus_endung["ausreisser"] / df_aus_endung["paare"]).round(3)
print("=== Ausreißer (> 2 s) je Endung ===")
print(df_aus_endung.to_string(index=False))
_ = tabelle_speichern(df_aus_endung, "02_14_dauer_ausreisser_endung")

# --- Richtung und Verhältnis -------------------------------------------------------
aus = df_ref_aus[df_ref_aus["ausreisser"]]
print("\n=== Richtung ===")
print(aus["richtung"].value_counts().to_string())
print("\n=== Verhältnis Serato / Datei (Quantile) ===")
print(aus["verhaeltnis"].quantile([0, 0.1, 0.25, 0.5, 0.75, 0.9, 1]).round(3).to_string())
df_aus_richtung = (aus.groupby("richtung", as_index=False)
                   .agg(anzahl=("pfad", "size"),
                        verhaeltnis_median=("verhaeltnis", "median")))
_ = tabelle_speichern(df_aus_richtung.round(3), "02_14_dauer_ausreisser_richtung")

aus.to_csv(ERR / "02_14_dauer_ausreisser.csv", index=False, encoding="utf-8")   # privat

In [ ]:
# =============================================================================
# Zelle 15 – Evidenz: 10 nicht regelbasiert zuordenbare vs. 10 zuordenbare Pfade
# =============================================================================
# Zweck: Anschauung, wie Pfade aussehen, die Stufe 1–2 NICHT lösen (Fehltreffer,
#   Zelle 06), im Vergleich zu Pfaden, die eindeutig gelöst werden.
# Auswahl proportional zu den Gruppen (SEED):
#   nicht zuordenbar (404): a 4 | d 1 | e 5
#   zuordenbar (2.350):     Stufe 1: 9 | Stufe 2: 1
# Je Pfad: was die Regeln finden (Index-Partner / Namenskandidat / nichts) und
#   was Serato noch weiß (database V2: Titel, Interpret, Dauer) + Datei-Dauer.
# Datenschutz: Pfade nur in der Ausgabe (nbstripout entfernt sie beim Commit).
# Voraussetzung: Zellen 01–14.
# =============================================================================


def serato_info(p: str) -> str:
    k = pfad_schluessel(p)
    if k not in db_idx.index:
        return "kein Eintrag in database V2"
    z = db_idx.loc[k]
    return f"Titel={z.get('tsng')!r} | Interpret={z.get('tart')!r} | Dauer={z.get('tlen')!r}"


def datei_dauer(index_pfad: str) -> str:
    d = tag_dauer.get(nfc(index_pfad))
    return f"{d:.2f} s" if d is not None and pd.notna(d) else "–"


# --- Auswahl ---------------------------------------------------------------------
offen = pd.concat([
    df_rest[df_rest["kategorie"] == KAT_A].sample(4, random_state=SEED),
    df_rest[df_rest["kategorie"] == KAT_D].sample(1, random_state=SEED),
    df_rest[df_rest["kategorie"] == KAT_E].sample(5, random_state=SEED),
])
geloest = pd.concat([
    df_pfade[df_pfade["treffer"] == "1 exakt"].sample(9, random_state=SEED),
    df_pfade[df_pfade["treffer"] == "2 erst nach NFC"].sample(1, random_state=SEED),
])

# --- Ausgabe: nicht zuordenbar -------------------------------------------------------
print("=" * 90)
print("A) NICHT regelbasiert zuordenbar (Stufe 1–2 ohne Treffer)")
print("=" * 90)
for nr, (_, z) in enumerate(offen.iterrows(), start=1):
    print(f"\n[A{nr}] Kategorie: {z['kategorie']}")
    print(f"   Crate-Pfad : {z['pfad']}")
    if z["n_kandidaten"] == 1:
        k = z["kandidaten"][0]
        print(f"   Kandidat   : {k}")
        print(f"   Datei-Dauer: {datei_dauer(k)}")
    else:
        print(f"   Kandidat   : keiner (Ordner im Index {'vorhanden' if z['kategorie'] == KAT_D else 'nicht vorhanden'})")
    print(f"   Serato     : {serato_info(z['pfad'])}")

# --- Ausgabe: zuordenbar --------------------------------------------------------------
print("\n" + "=" * 90)
print("B) Regelbasiert eindeutig zuordenbar (Stufe 1–2)")
print("=" * 90)
for nr, (_, z) in enumerate(geloest.iterrows(), start=1):
    partner = index_partner(z["pfad"], z["treffer"])
    print(f"\n[B{nr}] {z['treffer']} | Kodierung: {z['klasse']}")
    print(f"   Crate-Pfad : {z['pfad']}")
    print(f"   Index-Pfad : {partner}")
    print(f"   Zeichen-identisch: {z['pfad'] == partner}")
    print(f"   Datei-Dauer: {datei_dauer(partner)} | Serato: {serato_info(z['pfad'])}")

In [ ]:
# --- Nutzerangaben zu Teil A als Gold-Labels (privat, data/gold) -----------------
# Angaben Projektinhaber 11.11.2026; nur Nummern im Code (P11), Pfade aus der Stichprobe.
NUTZER_ANGABEN = {
    "A1": ("richtige Datei", "Ordner nachträglich bewusst angelegt"),
    "A2": ("richtige Datei", "Ordner versehentlich verschoben"),
    "A3": ("richtige Datei", "Ordner nachträglich bewusst angelegt"),
    "A4": ("unbekannt", ""), "A5": ("unbekannt", ""), "A6": ("unbekannt", ""),
    "A7": ("unbekannt", ""), "A8": ("unbekannt", ""), "A9": ("unbekannt", ""),
    "A10": ("unbekannt", ""),
}
df_nutzer_labels = offen[["pfad", "kategorie"]].reset_index(drop=True)
df_nutzer_labels.insert(0, "nr", [f"A{i}" for i in range(1, len(df_nutzer_labels) + 1)])
df_nutzer_labels["kandidat"] = offen["kandidaten"].map(lambda k: k[0] if k else None).to_numpy()
df_nutzer_labels["label"] = df_nutzer_labels["nr"].map(lambda n: NUTZER_ANGABEN[n][0])
df_nutzer_labels["begruendung"] = df_nutzer_labels["nr"].map(lambda n: NUTZER_ANGABEN[n][1])
df_nutzer_labels["quelle"] = "Nutzerangabe 11.11.2026"
df_nutzer_labels.to_csv(GOLD / "02_15_nutzer_labels.csv", index=False, encoding="utf-8")
print(df_nutzer_labels["label"].value_counts().to_string())

### Befund Zellen 13–15 – Inhaltsbeleg Dauer und Anschauung

**Kalibrierung Dauer (Zelle 13), |Serato − Datei|:**
- Referenz (Stufe 1/2, Pfad = Datei, n = 2.264): 92,1 % ≤ 0,1 s | 96,6 % ≤ 1 s | 96,8 % ≤ 2 s.
- Kategorie a (Namenskandidat, n = 87): 83,9 % ≤ 0,1 s | 98,9 % ≤ 1 s.

**Ausreißer (Zelle 14):** 72 Referenzpaare (3,2 %) weichen um mehr als 2 s ab, alle mp3 (wav/aif: nur 4 Referenzpaare, keine Aussage).
Serato länger in 55, Datei länger in 17 Fällen; Verhältnis Serato/Datei 0,04–35,9 (Median 1,93) → kein einheitlicher Faktor.
Welche Seite falsch ist, ist mit zwei Messwegen nicht entscheidbar.
**Entscheidung 11.11.2026:** Ein dritter Messweg (foobar als Fremdsystem) wird auf Phase 4 verschoben (Kalibrierung von Vetos).
Bis dahin gilt: Eine Dauer-Abweichung > 2 s ist **kein** hartes Veto.

**Anschauung (Zelle 15), 10 nicht zuordenbare vs. 10 zuordenbare Pfade:**
- 3 von 4 Kandidaten der Kategorie a sind laut Nutzer richtig (bewusst angelegter bzw. versehentlich verschobener Ordner); Dauer Δ 0,00 / 0,00 / 0,89 s.
- 1 Kandidat ohne Serato-Eintrag ist nur über Namensgleichheit belegt; die Ordnernamen weichen geringfügig ab → Vorschlag, nicht sicher.
- d/e (6 Beispiele): keine Datei gefunden; vorhanden sind nur Serato-Metadaten.
- **Neuer Befund:** Das Serato-Feld Titel enthält häufig DJ-Notizen (Präfixe wie Zeitmarken oder Kürzel), gekürzte oder nichtssagende Titel → ein Titelvergleich muss unscharf sein (Phase 4). Häufigkeit nicht gemessen.
- Gold-Labels (privat): 3 × „richtige Datei", 7 × „unbekannt" (`data/gold/02_15_nutzer_labels.csv`).

In [ ]:
# =============================================================================
# Zelle 16 – Übrige Playlist-Quellen: Format-Fingerabdruck (M3U, WPL, foobar)
# =============================================================================
# Zweck: Vor dem Parser messen, wie die Dateien aufgebaut sind (keine Annahmen):
#   Kodierung (BOM / UTF-8 / cp1252), Zeilenende, Kopfzeile, EXTINF, Pfadform.
# Quellen (Snapshot data/raw/playlists): mac_playlists, musik_playlists,
#   foobar_asus, foobar_medion. AppleDouble (._*) nur gezählt, nicht gelesen.
# Pfadform (Rechenvorschrift je Eintrag):
#   Windows absolut  ^[A-Za-z]:[\\/] | Mac /Volumes/ | Unix absolut ^/
#   URL ^file:// | relativ: sonst
#   M3U: jede nicht leere Zeile ohne '#'; WPL: Attribut src="..." in <media>.
# Kodierung: BOM UTF-8 / UTF-16 -> entsprechend; sonst UTF-8 streng versucht,
#   bei Fehler cp1252 (Befund foobar .m3u, Zelle 05).
# Datenschutz: Einträge privat (data/interim), public nur Zählwerte.
# =============================================================================

import re

PL = RAW / "playlists"
PFADFORMEN = [("Windows absolut", re.compile(r"^[A-Za-z]:[\\/]")),
              ("Mac /Volumes/", re.compile(r"^/Volumes/")),
              ("Unix absolut", re.compile(r"^/")),
              ("URL file://", re.compile(r"^file://", re.I))]
WPL_SRC = re.compile(r'<media[^>]*\bsrc="([^"]*)"', re.I)


def pfadform(p: str) -> str:
    for name, muster in PFADFORMEN:
        if muster.match(p):
            return name
    return "relativ"


def dekodieren(b: bytes) -> tuple[str, str]:
    if b.startswith(b"\xef\xbb\xbf"):
        return b[3:].decode("utf-8", errors="replace"), "UTF-8 mit BOM"
    if b.startswith((b"\xff\xfe", b"\xfe\xff")):
        return b.decode("utf-16"), "UTF-16 mit BOM"
    try:
        return b.decode("utf-8"), ("ASCII" if b.isascii() else "UTF-8 ohne BOM")
    except UnicodeDecodeError:
        return b.decode("cp1252"), "cp1252"


dateien, eintraege_pl = [], []
for quelle_dir in sorted(p for p in PL.iterdir() if p.is_dir()):
    for f in sorted(quelle_dir.rglob("*")):
        if not f.is_file() or f.name == ".gitkeep":
            continue
        if f.name.startswith("._"):
            dateien.append({"quelle": quelle_dir.name, "endung": f.suffix.lower(),
                            "appledouble": True})
            continue
        b = f.read_bytes()
        text, kod = dekodieren(b)
        zeilen_txt = text.splitlines()
        if f.suffix.lower() == ".wpl":
            pfade = WPL_SRC.findall(text)
        else:
            pfade = [z.strip() for z in zeilen_txt if z.strip() and not z.lstrip().startswith("#")]
        dateien.append({
            "quelle": quelle_dir.name, "datei": f.relative_to(quelle_dir).as_posix(),
            "endung": f.suffix.lower(), "appledouble": False, "kodierung": kod,
            "zeilenende": ("CRLF" if b"\r\n" in b else "LF" if b"\n" in b else
                           "CR" if b"\r" in b else "keins"),
            "kopf_extm3u": text.lstrip("\ufeff").startswith("#EXTM3U"),
            "n_extinf": sum(z.startswith("#EXTINF") for z in zeilen_txt),
            "n_eintraege": len(pfade)})
        eintraege_pl += [{"quelle": quelle_dir.name, "datei": f.relative_to(quelle_dir).as_posix(),
                          "endung": f.suffix.lower(), "eintrag": p, "pfadform": pfadform(p)}
                         for p in pfade]

df_pl_dateien = pd.DataFrame(dateien)
df_pl_eintraege = pd.DataFrame(eintraege_pl)
df_pl_dateien.to_csv(PROJECT_ROOT / "data/interim/02_16_playlist_dateien.csv", index=False, encoding="utf-8")
df_pl_eintraege.to_csv(PROJECT_ROOT / "data/interim/02_16_playlist_eintraege.csv", index=False, encoding="utf-8")

# --- 1. Dateien je Quelle ----------------------------------------------------------
print("=== 1: Dateien je Quelle und Endung ===")
print(df_pl_dateien.groupby(["quelle", "endung", "appledouble"]).size().to_string())
print(f"Summe: {len(df_pl_dateien)} (Snapshot: 213)")

# --- 2. Format-Fingerabdruck (ohne AppleDouble) --------------------------------------
echt = df_pl_dateien[~df_pl_dateien["appledouble"]]
df_pl_format = (echt.groupby(["quelle", "endung", "kodierung", "zeilenende", "kopf_extm3u"],
                             as_index=False)
                .agg(dateien=("datei", "size"), mit_extinf=("n_extinf", lambda s: (s > 0).sum()),
                     eintraege=("n_eintraege", "sum"), leer=("n_eintraege", lambda s: (s == 0).sum())))
print("\n=== 2: Format-Fingerabdruck ===")
print(df_pl_format.to_string(index=False))
_ = tabelle_speichern(df_pl_format, "02_16_playlist_format")

# --- 3. Pfadform der Einträge -----------------------------------------------------------
df_pl_pfadform = pd.crosstab([df_pl_eintraege["quelle"], df_pl_eintraege["endung"]],
                             df_pl_eintraege["pfadform"], margins=True, margins_name="Summe")
print("\n=== 3: Pfadform der Einträge ===")
print(df_pl_pfadform.to_string())
_ = tabelle_speichern(df_pl_pfadform.reset_index(), "02_16_playlist_pfadform")


In [ ]:
# =============================================================================
# Zelle 17 – Übrige Quellen: kanonische Pfade und Trefferstufen im Index
# =============================================================================
# Kanonisierung je Pfadform (Rechenvorschrift):
#   URL file://      Prozent-Dekodierung; '/Volumes/<Datenträger>/' entfernen
#   Windows absolut  Laufwerk 'X:\' entfernen, '\' -> '/'
#   relativ          relativ zum Ordner der Playlist-Datei auf dem Stick
#                    (Hypothese; geprüft über die Trefferquote), '..' auflösen
# Trefferstufen (erste zutreffende): 1 exakt | 2 nach NFC |
#   2b nach NFC + Kleinschreibung (FAT32/Windows unterscheiden Groß/Klein nicht) |
#   3 ohne Akzente | 4 kein Treffer.
# Zusatz: Laufwerksbuchstaben und Datenträgernamen zählen (Fremdsystem-Pfade).
# Voraussetzung: Zellen 01–16 (idx_exakt, idx_nfc, idx_akzentfrei, akzente_entfernen, nfc).
# =============================================================================

import posixpath
from urllib.parse import unquote, urlparse

BASIS = {"musik_playlists": cfg["quellen"]["musik"],
         "mac_playlists": cfg["quellen"]["mac_playlists"],
         "foobar_asus": cfg["laptops"][0]["zielordner"],
         "foobar_medion": cfg["laptops"][1]["zielordner"]}
idx_lower = {p.lower() for p in idx_nfc}


def kanonisch(eintrag: str, form: str, quelle: str, datei: str) -> tuple[str, str]:
    """Liefert (kanonischer Pfad relativ zur Stick-Wurzel, Herkunftsmerkmal)."""
    if form == "URL file://":
        p = unquote(urlparse(eintrag).path)
        teile = p.split("/")                         # ['', 'Volumes', '<Name>', ...]
        if len(teile) > 3 and teile[1] == "Volumes":
            return "/".join(teile[3:]), f"Volumes/{teile[2]}"
        return p.lstrip("/"), "URL ohne Volumes"
    if form == "Windows absolut":
        return eintrag[3:].replace("\\", "/"), f"Laufwerk {eintrag[:2].upper()}"
    if form == "relativ":
        ordner_pl = posixpath.dirname(f"{BASIS[quelle]}/{datei}")
        return posixpath.normpath(posixpath.join(ordner_pl, eintrag.replace("\\", "/"))), "relativ"
    return eintrag, form


def stufe(p: str) -> str:
    if p in idx_exakt:
        return "1 exakt"
    q = nfc(p)
    if q in idx_nfc:
        return "2 nach NFC"
    if q.lower() in idx_lower:
        return "2b nach NFC + Kleinschreibung"
    if akzente_entfernen(p) in idx_akzentfrei:
        return "3 ohne Akzente"
    return "4 kein Treffer"


erg = [kanonisch(e, f, q, d) for e, f, q, d in
       zip(df_pl_eintraege["eintrag"], df_pl_eintraege["pfadform"],
           df_pl_eintraege["quelle"], df_pl_eintraege["datei"])]
df_pl_eintraege["kanonisch"] = [e[0] for e in erg]
df_pl_eintraege["herkunft"] = [e[1] for e in erg]
df_pl_eintraege["stufe"] = df_pl_eintraege["kanonisch"].map(stufe)
df_pl_eintraege.to_csv(PROJECT_ROOT / "data/interim/02_17_playlist_eintraege_kanonisch.csv",
                       index=False, encoding="utf-8")                    # privat

# --- 1. Herkunftsmerkmal (Laufwerke, Datenträgernamen) ------------------------------
print("=== 1: Herkunft der absoluten Pfade ===")
print(df_pl_eintraege.groupby(["quelle", "herkunft"]).size().to_string())
_ = tabelle_speichern(df_pl_eintraege.groupby(["quelle", "herkunft"], as_index=False)
                      .size().rename(columns={"size": "eintraege"}), "02_17_playlist_herkunft")

# --- 2. Trefferstufen je Quelle und Pfadform -------------------------------------------
df_pl_stufen = pd.crosstab([df_pl_eintraege["quelle"], df_pl_eintraege["pfadform"]],
                           df_pl_eintraege["stufe"], margins=True, margins_name="Summe")
print("\n=== 2: Trefferstufen (Einträge) ===")
print(df_pl_stufen.to_string())
_ = tabelle_speichern(df_pl_stufen.reset_index(), "02_17_playlist_trefferstufen")

In [ ]:
# =============================================================================
# Zelle 18 – Fehltreffer übriger Quellen: Dateiname im Ordner der Playlist?
# =============================================================================
# Hypothese (Zelle 17): Playlists im Musikordner sind überwiegend Album-Playlists
#   fremder Rechner (Laufwerke C:, I:, Q: ...); die Tracks liegen im selben Ordner
#   wie die Playlist-Datei.
# Rechenvorschrift je Fehltreffer (Stufe 4), Schlüssel k = lower(NFC(Dateiname)),
#   Dateiname = letzter Teil nach '/' oder '\', erste zutreffende Kategorie:
#   P  Name im Ordner der Playlist-Datei (genau dort im Index)
#   A1 Name genau 1x anderswo im Index
#   AM Name mehrfach anderswo
#   N  Name nicht im Index
# Namensgleichheit ≠ Inhaltsgleichheit (Zelle 06): P/A1/AM sind Kandidaten, keine Treffer.
# Voraussetzung: Zellen 01–17 (df_audio, df_pl_eintraege, BASIS, nfc).
# =============================================================================

name_key = lambda p: nfc(re.split(r"[\\/]", p)[-1]).lower()

ordner_namen = (df_audio.assign(o=df_audio["pfad_relativ"].map(lambda p: nfc(posixpath.dirname(p))),
                                k=df_audio["pfad_relativ"].map(name_key))
                .groupby("o")["k"].apply(set).to_dict())
name_anzahl = df_audio["pfad_relativ"].map(name_key).value_counts().to_dict()


def kategorie_pl(z) -> str:
    k = name_key(z["eintrag"])
    ordner_pl = nfc(posixpath.dirname(f"{BASIS[z['quelle']]}/{z['datei']}"))
    if k in ordner_namen.get(ordner_pl, set()):
        return "P Name im Ordner der Playlist"
    n = name_anzahl.get(k, 0)
    return "A1 Name 1x anderswo" if n == 1 else "AM Name mehrfach anderswo" if n > 1 else "N Name nicht im Index"


rest_pl = df_pl_eintraege[df_pl_eintraege["stufe"] == "4 kein Treffer"].copy()
rest_pl["kategorie"] = rest_pl.apply(kategorie_pl, axis=1)
rest_pl.to_csv(ERR / "02_18_playlist_fehltreffer.csv", index=False, encoding="utf-8")   # privat

df_pl_rest_kat = pd.crosstab([rest_pl["quelle"], rest_pl["pfadform"]], rest_pl["kategorie"],
                             margins=True, margins_name="Summe")
print("=== Fehltreffer übriger Quellen nach Kategorie (Einträge) ===")
print(df_pl_rest_kat.to_string())
_ = tabelle_speichern(df_pl_rest_kat.reset_index(), "02_18_playlist_fehltreffer_kategorie")

In [ ]:
# =============================================================================
# Zelle 19 – Kategorie A1 übriger Quellen: gleiches Pfadende, anderer Präfix?
# =============================================================================
# Befund Zelle 18: P = 0 (Hypothese "Album-Playlist im selben Ordner" widerlegt);
#   3.000 Fehltreffer mit genau 1 Namenskandidaten anderswo (A1).
# Neue Hypothese: Pfadende (Ordner + Datei) stimmt, nur der Präfix (Wurzel) ist
#   anders; der Präfix-Tausch ist je Playlist-Datei einheitlich -> deterministische
#   Regel "Präfix ersetzen" möglich.
# Rechenvorschrift je A1-Eintrag:
#   Teile = Pfad in Komponenten (NFC, Kleinschreibung, Trenner '/' oder '\')
#   gleiche_endteile = Anzahl übereinstimmender Komponenten von hinten
#   praefix_alt = Eintrag ohne gleiche Endteile, praefix_neu = Kandidat ohne diese
# Je Playlist-Datei: Anzahl verschiedener Präfix-Paare (1 = einheitlich).
# Voraussetzung: Zellen 01–18 (rest_pl, df_audio, name_key, nfc).
# =============================================================================

name_zu_pfad = (df_audio.assign(k=df_audio["pfad_relativ"].map(name_key))
                .drop_duplicates("k", keep=False).set_index("k")["pfad_relativ"].to_dict())


def teile(p: str) -> list[str]:
    return [t for t in re.split(r"[\\/]", nfc(p).lower()) if t]


def endvergleich(eintrag: str, kandidat: str) -> tuple[int, str, str]:
    a, b = teile(eintrag), teile(kandidat)
    n = 0
    while n < min(len(a), len(b)) and a[-1 - n] == b[-1 - n]:
        n += 1
    return n, "/".join(a[:len(a) - n]), "/".join(b[:len(b) - n])


a1 = rest_pl[rest_pl["kategorie"] == "A1 Name 1x anderswo"].copy()
a1["kandidat"] = a1["eintrag"].map(lambda e: name_zu_pfad[name_key(e)])
erg = [endvergleich(e, k) for e, k in zip(a1["eintrag"], a1["kandidat"])]
a1["gleiche_endteile"] = [e[0] for e in erg]
a1["praefix_alt"] = [e[1] for e in erg]
a1["praefix_neu"] = [e[2] for e in erg]
a1.to_csv(ERR / "02_19_playlist_a1_praefix.csv", index=False, encoding="utf-8")   # privat

# --- 1. Wie viele Endteile stimmen überein? ------------------------------------------
df_endteile = pd.crosstab(a1["pfadform"], a1["gleiche_endteile"].clip(upper=4),
                          margins=True, margins_name="Summe")
print("=== 1: übereinstimmende Endteile (1 = nur Dateiname, 4 = 4 oder mehr) ===")
print(df_endteile.to_string())
_ = tabelle_speichern(df_endteile.reset_index(), "02_19_playlist_a1_endteile")

# --- 2. Präfix-Paare je Playlist-Datei ------------------------------------------------
je_datei = (a1.groupby(["quelle", "datei"])
            .agg(eintraege=("eintrag", "size"),
                 praefix_paare=("praefix_alt", lambda s: a1.loc[s.index, ["praefix_alt", "praefix_neu"]]
                                .drop_duplicates().shape[0])))
df_praefix = (je_datei["praefix_paare"].clip(upper=3).value_counts().sort_index()
              .rename_axis("praefix_paare_je_datei (3 = 3+)").reset_index(name="playlist_dateien"))
df_praefix["eintraege"] = [je_datei.loc[je_datei["praefix_paare"].clip(upper=3) == v, "eintraege"].sum()
                           for v in df_praefix.iloc[:, 0]]
print("\n=== 2: verschiedene Präfix-Paare je Playlist-Datei ===")
print(df_praefix.to_string(index=False))
_ = tabelle_speichern(df_praefix, "02_19_playlist_a1_praefix_je_datei")


### Befund Zellen 16–19 – Übrige Playlist-Quellen (M3U, WPL, foobar)

**Format (Zelle 16):** 213 Dateien (26 AppleDouble), **10.289 Einträge** (mehr als alle Crates: 7.915).
Kodierung gemischt (ASCII, UTF-8 ohne BOM, nicht UTF-8 → als cp1252 gelesen); kein BOM; CRLF überwiegt.
Pfadformen: `file://`-URL 495 (nur Mac) | Windows absolut 2.747 | relativ 7.047.

**Trefferquote Stufe 1+2 (Zelle 17):** foobar_asus 197/197 (100 %) | foobar_medion 322/325 | Mac 481/502 (95,8 %) | Musikordner relativ 5.347/7.047 (75,9 %) | **Musikordner Windows absolut 0/2.218** (Laufwerke C:, D:, I:, Q: fremder Rechner). Stufe 2b (Groß/Klein) und Stufe 3 (Akzente) = 0.

**Fehltreffer (Zelle 18, 3.942):** Hypothese „Tracks im Ordner der Playlist" widerlegt (0 Fälle). Name genau 1× anderswo 3.000 | mehrfach 48 | nicht im Index 894.

**Pfadende (Zelle 19, 3.000 mit genau 1 Kandidaten):** Bei Windows-absoluten Pfaden stimmen in 1.750 von 1.974 Fällen (88,7 %) **Ordner und Dateiname** überein → mögliche deterministische Regel „eindeutiges Pfadende aus ≥ 2 Teilen" (zu prüfen am Gold-Set, Phase 3/4). Relative Fehltreffer: überwiegend nur Dateiname gleich (804/1.026).
Die Kennzahl „Präfix-Paare je Playlist-Datei" ist durch die Rechenvorschrift verfälscht (unterschiedlich viele gleiche Endteile erzeugen verschiedene Paare) → keine Aussage; Präfix-Regel wird im Resolver-Design geprüft.

**Offen:** Ob „cp1252" stimmt oder ein anderer 1-Byte-Zeichensatz vorliegt (z. B. MacRoman) – bisher keine Treffer der Stufe 3, kein Hinweis auf falsche Dekodierung.

In [ ]:
# =============================================================================
# Zelle 20 – Dubletten: Vorauswahl über gleiche Dateigröße (ohne Stick-Zugriff)
# =============================================================================
# Begründung: Identischer Inhalt setzt identische Größe voraus. Nur Dateien,
#   deren Größe mindestens 2x vorkommt, müssen gehasht werden.
# Ergebnis: Anzahl Kandidaten und zu lesende Datenmenge -> Entscheidung, ob
#   der Hash-Lauf (nächste Zelle, liest vom Stick) verhältnismäßig ist.
# Zusatz: Gruppen gleicher Größe UND gleichen Dateinamens (Hinweis, kein Beleg).
# Voraussetzung: Zellen 01–19 (df_audio mit Spalte 'bytes', name_key).
# =============================================================================

groesse_n = df_audio["bytes"].map(df_audio["bytes"].value_counts())
df_dup_kand = df_audio[groesse_n >= 2].copy()
df_dup_kand["name_k"] = df_dup_kand["pfad_relativ"].map(name_key)

n_gruppen = df_dup_kand["bytes"].nunique()
n_gleich_name = (df_dup_kand.groupby(["bytes", "name_k"]).size() >= 2).sum()

df_dup_vorauswahl = pd.DataFrame([
    {"kennzahl": "Audiodateien gesamt", "wert": len(df_audio)},
    {"kennzahl": "Dateien mit Größe >= 2x", "wert": len(df_dup_kand)},
    {"kennzahl": "Größengruppen", "wert": n_gruppen},
    {"kennzahl": "Gruppen gleiche Größe + gleicher Name", "wert": int(n_gleich_name)},
    {"kennzahl": "zu lesende GB", "wert": round(df_dup_kand["bytes"].sum() / 1e9, 2)},
    {"kennzahl": "Anteil an 128,4 GB", "wert": round(df_dup_kand["bytes"].sum() / df_index["bytes"].sum(), 3)},
])
print(df_dup_vorauswahl.to_string(index=False))
print("\nKandidaten je Endung:")
print(df_dup_kand["endung"].value_counts().to_string())
_ = tabelle_speichern(df_dup_vorauswahl, "02_20_dubletten_vorauswahl")
df_dup_kand.to_csv(PROJECT_ROOT / "data/interim/02_20_dubletten_kandidaten.csv",
                   index=False, encoding="utf-8")                        # privat


In [ ]:
# =============================================================================
# Zelle 21 – Dubletten: SHA-256 der Kandidaten gleicher Größe (nur lesend vom Stick)
# =============================================================================
# Vorauswahl Zelle 20: 190 Dateien in 93 Größengruppen, 2,9 GB (2,3 % des Sticks).
# Rechenvorschrift: Dublette = gleicher SHA-256 (bitgleicher Inhalt).
# Auswertung: Dubletten-Gruppen; gleicher vs. verschiedener Name;
#   Bezug zur Policy-Frage "mehrere Kandidaten": Sind die Kandidaten der
#   Kategorie AM (Zelle 18, Name mehrfach im Index) inhaltlich identisch?
# Datenschutz: Pfade privat (data/interim), public nur Zählwerte.
# Voraussetzung: Zellen 01–20, Stick eingesteckt.
# =============================================================================

import hashlib


def sha256_datei(pfad: Path) -> str:
    h = hashlib.sha256()
    with open(pfad, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


assert (STICK_WURZEL / "Musik").is_dir(), f"Stick nicht unter {STICK_LW} gefunden – einstecken."
t0 = time.perf_counter()
df_dup_kand["sha256"] = df_dup_kand["pfad_relativ"].map(lambda p: sha256_datei(STICK_WURZEL / p))
laufzeit_s = time.perf_counter() - t0
df_dup_kand.to_csv(PROJECT_ROOT / "data/interim/02_21_dubletten_hash.csv",
                   index=False, encoding="utf-8")                        # privat

# --- 1. Dubletten-Gruppen ----------------------------------------------------------
grp = df_dup_kand.groupby("sha256").agg(dateien=("pfad_relativ", "size"),
                                        namen=("name_k", "nunique"))
dup = grp[grp["dateien"] >= 2]
df_dubletten = pd.DataFrame([
    {"kennzahl": "Laufzeit Hash (s)", "wert": round(laufzeit_s, 1)},
    {"kennzahl": "Durchsatz (MB/s)", "wert": round(df_dup_kand["bytes"].sum() / 1e6 / laufzeit_s, 1)},
    {"kennzahl": "Dubletten-Gruppen (gleicher Hash)", "wert": len(dup)},
    {"kennzahl": "Dateien in Dubletten-Gruppen", "wert": int(dup["dateien"].sum())},
    {"kennzahl": "überzählige Kopien", "wert": int(dup["dateien"].sum() - len(dup))},
    {"kennzahl": "Gruppen mit gleichem Namen", "wert": int((dup["namen"] == 1).sum())},
    {"kennzahl": "Gruppen mit verschiedenen Namen", "wert": int((dup["namen"] > 1).sum())},
    {"kennzahl": "MB überzählig", "wert": round(
        df_dup_kand[df_dup_kand["sha256"].isin(dup.index)]
        .groupby("sha256")["bytes"].apply(lambda s: s.iloc[1:].sum()).sum() / 1e6, 1)},
])

# --- 2. Kategorie AM: sind die mehrfachen Namenskandidaten identisch? ----------------
hash_von = dict(zip(df_dup_kand["pfad_relativ"], df_dup_kand["sha256"]))
pfade_je_name = df_audio.assign(k=df_audio["pfad_relativ"].map(name_key)).groupby("k")["pfad_relativ"].apply(list)
am = rest_pl[rest_pl["kategorie"] == "AM Name mehrfach anderswo"]
am_identisch = [len({hash_von.get(p) for p in pfade_je_name[name_key(e)]}) == 1
                and all(p in hash_von for p in pfade_je_name[name_key(e)])
                for e in am["eintrag"]]
df_dubletten = pd.concat([df_dubletten, pd.DataFrame([
    {"kennzahl": "AM-Einträge (Zelle 18)", "wert": len(am)},
    {"kennzahl": "AM: alle Kandidaten bitgleich", "wert": int(sum(am_identisch))},
])], ignore_index=True)
print(df_dubletten.to_string(index=False))
_ = tabelle_speichern(df_dubletten, "02_21_dubletten")

### Befund Zellen 20–21 – Dubletten (bitgleicher Inhalt)

**Vorauswahl (Zelle 20, ohne Stick-Zugriff):** Identischer Inhalt setzt gleiche Größe voraus → nur 190 von 5.236 Audiodateien (93 Größengruppen, 2,9 GB = 2,3 % des Sticks) kommen in Frage.

**Hash (Zelle 21, SHA-256, nur lesend):** 19,3 s (150 MB/s). 45 Dubletten-Gruppen, jede mit genau 2 Dateien; 45 überzählige Kopien (960 MB); 32 Gruppen mit gleichem, 13 mit verschiedenem Dateinamen.

**Bezug zur Policy „mehrere Kandidaten":** Von 48 Einträgen mit mehrfachem Namenskandidaten (Zelle 18) sind bei 15 alle Kandidaten bitgleich (Wahl ohne Einfluss auf den Inhalt), bei 33 nicht (echte Auswahl). Vorschlag für Phase 3 (Entscheidung offen): bitgleich → sicher, sonst → Vorschlag.

**Machbarkeit Hash des ganzen Sticks:** bei gemessenem Durchsatz ca. 14 min (Workflow-Schritt 2, Index mit Hash).

Dubletten werden nur gemeldet, nie gelöscht (P1).

### Entscheidung 12.11.2026 – Zielformat foobar

Das Zielformat ist durch Daten belegt, ein weiterer Test ist nicht nötig:
- foobar Asus: 197/197 Einträge absolut `D:\Musik\...`, alle Treffer (Zelle 17).
- foobar Medion: 325 Einträge absolut `E:\Musik\...`, 322 Treffer (Zelle 17).
- Angabe Projektinhaber: relative Pfade haben in foobar nicht funktioniert.

**Folge für den Writer (Phase 3):** je Ziel-Laptop eine eigene Playlist im Zielordner auf dem Stick (`Playlist_AsusExpertBook_foobar2000` bzw. `Playlist_Medion_foobar2000`) mit absoluten Pfaden und dem Laufwerk des Laptops (Asus `D:`, Medion `E:`, Zuordnung über Hostname). Kodierung wie von foobar geschrieben (Zelle 05): `.m3u` cp1252 oder `.m3u8` UTF-8 mit BOM, CRLF.

In [ ]:
# =============================================================================
# Zelle 23 – EDA-Abschluss: Schichten für Gold-Set und Grundlage der Split-Methode
# =============================================================================
# Gold-Set = Messmittel für Precision der Klasse "sicher" (P5). Schichten aus
#   den EDA-Befunden (Rechenvorschriften der Zellen 03, 06, 17–21).
# Umfang je Schicht nach Rule of Three: 0 Fehler in n Fällen -> Fehlerrate < 3/n
#   (95 %): n = 60 -> < 5 % | n = 100 -> < 3 % | n = 300 -> < 1 %;
#   ist die Schicht kleiner, wäre es eine Vollerhebung (n = N).
# Split-Grundlage: Konzentration je Ordner (Ordner des kanonischen Pfads).
#   Hohe Konzentration -> Gefahr, dass ein Modell Ordner-Muster auswendig lernt
#   -> gruppierter Split nach Ordner statt zufälliger Split.
# Zählung je eindeutigem NFC-Pfad (Eintrag-Mehrfachzählung vermieden, Zelle 03).
# Voraussetzung: Zellen 01–21.
# =============================================================================

am_ok = np.array(am_identisch, dtype=bool)
SCHICHTEN = [
    ("K1 Crates Stufe 1 exakt",           df_pfade.loc[df_pfade["treffer"] == "1 exakt", "pfad"]),
    ("K2 Crates Stufe 2 NFC",             df_pfade.loc[df_pfade["treffer"] == "2 erst nach NFC", "pfad"]),
    ("K3 Crates a: Name 1x anderswo",     df_rest.loc[df_rest["kategorie"] == KAT_A, "pfad"]),
    ("K4 Crates d/e: nur Metadaten",      df_rest.loc[df_rest["kategorie"].isin([KAT_D, KAT_E]), "pfad"]),
    ("Q1 übrige Stufe 1/2",               df_pl_eintraege.loc[df_pl_eintraege["stufe"].isin(["1 exakt", "2 nach NFC"]), "kanonisch"]),
    ("Q2 übrige A1: Ordner+Datei gleich", a1.loc[a1["gleiche_endteile"] >= 2, "kanonisch"]),
    ("Q3 übrige A1: nur Dateiname",       a1.loc[a1["gleiche_endteile"] == 1, "kanonisch"]),
    ("Q4a übrige AM: bitgleich",          am.loc[am_ok, "kanonisch"]),
    ("Q4b übrige AM: verschieden",        am.loc[~am_ok, "kanonisch"]),
    ("Q5 übrige N: Name nicht im Index",  rest_pl.loc[rest_pl["kategorie"] == "N Name nicht im Index", "kanonisch"]),
]

zeilen = []
for name, s in SCHICHTEN:
    eindeutig = pd.Series(s.map(nfc).unique())
    ordner_n = eindeutig.map(posixpath.dirname).value_counts()
    N = len(eindeutig)
    zeilen.append({
        "schicht": name, "eintraege": len(s), "eindeutig_N": N,
        "n_5pct": min(60, N), "n_3pct": min(100, N), "n_1pct": min(300, N),
        "ordner": len(ordner_n),
        "top1_ordner_anteil": round(ordner_n.iloc[0] / N, 3) if N else np.nan,
        "top5_ordner_anteil": round(ordner_n.iloc[:5].sum() / N, 3) if N else np.nan,
    })
df_schichten = pd.DataFrame(zeilen)
print(df_schichten.to_string(index=False))
print(f"\nSumme n bei < 5 % / < 3 % / < 1 %: "
      f"{df_schichten['n_5pct'].sum()} / {df_schichten['n_3pct'].sum()} / {df_schichten['n_1pct'].sum()}")
_ = tabelle_speichern(df_schichten, "02_23_goldset_schichten")

### Befund und Entscheidungen Zelle 23 (Entscheidung 12.11.2026)

**Ordner-Konzentration:** In den schweren Schichten stellt ein einzelner Ordner 41–49 % der Pfade (K2, K3, K4), die 5 größten Ordner 72–90 %; in K1 (exakt) dagegen 5,6 % bzw. 15 %.

**Entscheidung Split:** gruppiert nach Ordner des kanonischen Pfads (keine Pfade eines Ordners in Training und Test gleichzeitig; Leakage-Schutz). Gilt für lernende Komponenten (Paar-Klassifikator, Schwellen), Phase 2/4.

**Entscheidung Gold-Set:** zuerst Pilot mit 20 Labels (geschichtet aus K3, Q2, K4) mit gemessener Zeit je Label; danach Festlegung des Umfangs je Schicht nach Zeitbudget und Zielgenauigkeit (Rule of Three). Stichprobe mit Obergrenze je Ordner. Umsetzung in Phase 2.

**Priorität der Schichten:** K3, Q2, Q4a = Schichten, in denen neue Regeln „sicher" behaupten würden (entscheidend für P5); K1, K2, Q1 = Kontrolle (Pfad = Datei); K4, Q3, Q4b, Q5 = Messung des Recall.

## Notebook 02 – Abschluss: EDA (Phase 1)

**Datenbasis:** Snapshot `data/raw` (Crates, Subcrates, `database V2`, Playlists) + Index; Tags (tinytag) und Hash (SHA-256) nur lesend vom Stick.

**Kernergebnisse:**
| Thema | Ergebnis |
|---|---|
| Crates (Wahrheitsquelle) | 7.915 Einträge, 2.754 Pfade; Stufe 1+2 sicher 85,3 %; Stufe 3 (Akzente) 0 |
| Playlist-Ebene | 125/167 Crates (74,9 %) vollständig sicher; Mittel je Crate 93,2 % (KI 90,1–96,0 %) |
| Fehltreffer Crates (404) | jeder hat eine Belegquelle (Namenskandidat oder Serato-Metadaten) |
| Subcrates (2021) | zusätzliche Quelle, Varianten `_2021` (Entscheidung d) |
| `database V2` | TLV, 5.478 Tracks; Abdeckung 96,8 % / 100 %; Titel enthält DJ-Notizen |
| Audio-Tags | 5.236 Dateien, 0 Fehler; Dauer 100 %, Tracknummer nur 61 % |
| Inhaltsbeleg Dauer | 96,6 % ≤ 1 s bei gleicher Datei; 3,2 % mp3-Ausreißer (Ursache offen, Phase 4) |
| Übrige Quellen | 10.289 Einträge; foobar 100 % / 99,1 %, Mac 95,8 %, Musikordner relativ 75,9 %, fremde Windows-Pfade 0 % |
| Pfadende-Regel | 1.750 fremde Pfade mit eindeutigem Ordner+Datei-Kandidaten |
| Dubletten | 45 Paare bitgleich; bei 15/48 Mehrfach-Kandidaten bitgleich |
| Zielformat | absolut, Laufwerk je Laptop (Asus D:, Medion E:), Zielordner auf dem Stick |

**Antwort auf die Leitfrage (Stand Phase 1):** Die Stufen 1+2 (exakter Pfad) lösen 85 % der Crate-Pfade und 75 % der Crates vollständig. Weitere Fälle löst eine **deterministische Namenssuche** (wie die Suche im Explorer): eindeutiger Name bzw. eindeutiges Pfadende bei 166 Crate-Pfaden (a) und 3.000 Einträgen übriger Quellen (A1); offen ist nur, ob „sicher" oder „Vorschlag" (Precision am Gold-Set).
Der Raum für Inhaltsbelege, Fuzzy-, ML- und Agent-Verfahren ist damit klein: 48 Einträge mit mehrfachem Namen (15 davon bitgleich) sowie 238 Crate-Pfade und 894 Einträge, deren Dateiname nicht auf dem Stick existiert – vermutlich zum großen Teil gelöschte Dateien, die kein Verfahren findet (nur Bericht „fehlt").
Ergänzung Bericht (Phase 3): leere Ordner erkennen (Hinweis „Album vermutlich gelöscht"; Beobachtung Projektinhaber, im Index bisher unsichtbar).

**Entscheidungen Phase 1:** Subcrates Variante d (09.11.) | tinytag statt mutagen (11.11.) | Dauer-Abweichung kein hartes Veto bis Phase 4 (11.11.) | Zielformat absolut je Laptop (12.11.) | Split gruppiert nach Ordner, Gold-Set mit Pilot (12.11.).

**Offen für spätere Phasen:** Ursache NFC/NFD-Mischung; Dauer-Ausreißer (dritter Messweg); Policy mehrere Kandidaten (Vorschlag: bitgleich → sicher); Präfix-Regel fremder Pfade; Kodierung „cp1252" vs. MacRoman.

In [ ]:
# =============================================================================
# Zelle 24 – Evidenz: 10 nicht regelbasiert zuordenbare Einträge übriger Quellen
# =============================================================================
# Ergänzt Zelle 15 (Crates). Auswahl proportional aus 3.942 Fehltreffern (SEED):
#   Q2 A1 Ordner+Datei gleich 4 | Q3 A1 nur Dateiname 3 | AM 1 | N 2
# Je Eintrag: Quelle, Playlist-Datei, Originaleintrag, kanonischer Pfad,
#   Kandidat(en) im Index, übereinstimmende Endteile, Datei-Dauer.
# Datenschutz: Pfade nur in der Ausgabe (nbstripout entfernt sie beim Commit).
# Voraussetzung: Zellen 01–23.
# =============================================================================

auswahl_pl = pd.concat([
    a1[a1["gleiche_endteile"] >= 2].sample(4, random_state=SEED).assign(gruppe="Q2 A1 Ordner+Datei gleich"),
    a1[a1["gleiche_endteile"] == 1].sample(3, random_state=SEED).assign(gruppe="Q3 A1 nur Dateiname"),
    am.sample(1, random_state=SEED).assign(gruppe="AM Name mehrfach"),
    rest_pl[rest_pl["kategorie"] == "N Name nicht im Index"].sample(2, random_state=SEED).assign(gruppe="N Name nicht im Index"),
])

for nr, (_, z) in enumerate(auswahl_pl.iterrows(), start=1):
    print(f"\n[C{nr}] {z['gruppe']} | Quelle: {z['quelle']} | Pfadform: {z['pfadform']}")
    print(f"   Playlist-Datei : {z['datei']}")
    print(f"   Eintrag (roh)  : {z['eintrag']}")
    print(f"   kanonisch      : {z['kanonisch']}")
    if z["gruppe"].startswith(("Q2", "Q3")):
        print(f"   Kandidat       : {z['kandidat']}")
        print(f"   gleiche Endteile: {z['gleiche_endteile']} | Datei-Dauer: {datei_dauer(z['kandidat'])}")
    elif z["gruppe"].startswith("AM"):
        for k in pfade_je_name[name_key(z["eintrag"])]:
            print(f"   Kandidat       : {k} | Dauer: {datei_dauer(k)} | SHA-256: {hash_von.get(k, '–')[:12]}")
    else:
        print("   Kandidat       : keiner (Dateiname nicht im Index)")

In [ ]:
# =============================================================================
# Zelle 25 – 0-Byte-Dateien an Stelle von Album-Ordnern
# =============================================================================
# Anlass (Prüfung Projektinhaber, Fall C10): Ein Album-"Ordner" ist eine Datei
#   mit 0 Byte (PowerShell Mode ------, Length 0). Playlist-Einträge darunter
#   können nicht existieren.
# Rechenvorschrift: Z = alle 0-Byte-Dateien im Index (beliebige Endung).
#   Fehltreffer, deren Ordner (dirname des kanonischen Pfads, NFC) in Z liegt,
#   zeigen auf einen "Ordner", der eine leere Datei ist.
# Hypothese (ungeprüft): Dateisystemfehler oder Überschreiben -> Datenverlust.
# Datenschutz: Pfade privat (reports/errors), public nur Zählwerte.
# Voraussetzung: Zellen 01–24 (df_index, df_rest, rest_pl, nfc).
# =============================================================================

null_byte = df_index[df_index["bytes"] == 0]
Z = set(null_byte["pfad_relativ"].map(nfc))

crates_fehl = df_rest[df_rest["kategorie"].isin([KAT_D, KAT_E])]["pfad"]
uebrige_fehl = rest_pl[rest_pl["kategorie"] == "N Name nicht im Index"]["kanonisch"]
ordner_von = lambda p: posixpath.dirname(nfc(p))

treffer_crates = crates_fehl[crates_fehl.map(ordner_von).isin(Z)]
treffer_uebrige = uebrige_fehl[uebrige_fehl.map(ordner_von).isin(Z)]

df_null_byte = pd.DataFrame([
    {"kennzahl": "0-Byte-Dateien im Index", "wert": len(null_byte)},
    {"kennzahl": "davon AppleDouble", "wert": int(null_byte["appledouble"].sum())},
    {"kennzahl": "Crates d/e: Ordner ist 0-Byte-Datei (Pfade)", "wert": len(treffer_crates)},
    {"kennzahl": "übrige N: Ordner ist 0-Byte-Datei (Einträge)", "wert": len(treffer_uebrige)},
    {"kennzahl": "betroffene 0-Byte-'Ordner' gesamt",
     "wert": len(set(treffer_crates.map(ordner_von)) | set(treffer_uebrige.map(ordner_von)))},
])
print(df_null_byte.to_string(index=False))
print("\n0-Byte-Dateien je Endung:")
print(null_byte["endung"].value_counts().head(10).to_string())
_ = tabelle_speichern(df_null_byte, "02_25_null_byte_ordner")
null_byte.to_csv(ERR / "02_25_null_byte_dateien.csv", index=False, encoding="utf-8")   # privat

### Befund Zelle 25 – Album-„Ordner" als 0-Byte-Dateien (Entdeckung 12.11.2026)

**Anlass:** Prüfung durch Projektinhaber (Fall C10): Ein Album-Ordner ist eine Datei mit 0 Byte.
**Messung:** 23 Dateien mit 0 Byte im Index (keine AppleDouble, 22 ohne Endung). An der Stelle von **10 Album-Ordnern** steht eine 0-Byte-Datei gleichen Namens.
- Crate-Pfade ohne Namenskandidaten (d/e): **229 von 238 (96,2 %)** zeigen in einen solchen „Ordner".
- Übrige Quellen, Name nicht im Index: **509 von 894 (56,9 %)**.

**Korrektur früherer Befunde:** Kategorie e („Ordner nicht vorhanden") und die Hypothese „gelöscht" sind ungenau; der Ordnername existiert überwiegend als 0-Byte-Datei.

**Hypothese (ungeprüft):** Dateisystemschaden (FAT32). Die Audiodaten dieser Alben sind auf dem Stick für kein Verfahren erreichbar; Wiederherstellung nur aus Sicherungen oder per Datenrettung.

**Folgen:** (1) Linter-Regel (Phase 3): 0-Byte-Dateien an Stelle von Ordnern erkennen und im Bericht als „Album beschädigt/verloren" melden. (2) Empfehlung an den Nutzer: vollständige Sicherung des Sticks vor weiteren Schreibvorgängen; keine Reparatur ohne Sicherung (P1). (3) Leitfrage: Der größte Teil der nicht auffindbaren Fälle ist kein Zuordnungsproblem, sondern Datenverlust.

In [ ]:
# --- Liste der betroffenen "Ordner" (nur Ausgabe, privat) -------------------------
df_null_ordner = pd.DataFrame({"ordner": sorted(Z)})
df_null_ordner["crate_pfade"] = df_null_ordner["ordner"].map(
    treffer_crates.map(ordner_von).value_counts()).fillna(0).astype(int)
df_null_ordner["uebrige_eintraege"] = df_null_ordner["ordner"].map(
    treffer_uebrige.map(ordner_von).value_counts()).fillna(0).astype(int)
df_null_ordner["betroffen"] = (df_null_ordner["crate_pfade"] + df_null_ordner["uebrige_eintraege"]) > 0
df_null_ordner = df_null_ordner.sort_values(["betroffen", "crate_pfade", "uebrige_eintraege"],
                                            ascending=False)

pd.set_option("display.max_colwidth", 120)
print("\n=== 0-Byte-Dateien: mit Playlist-Bezug (Album-'Ordner') ===")
print(df_null_ordner[df_null_ordner["betroffen"]].drop(columns="betroffen").to_string(index=False))
print("\n=== 0-Byte-Dateien: ohne Playlist-Bezug ===")
print(df_null_ordner[~df_null_ordner["betroffen"]][["ordner"]].to_string(index=False))
df_null_ordner.to_csv(ERR / "02_25_null_byte_ordner_liste.csv", index=False, encoding="utf-8")   # privat

## Notebook 02 – Abschluss: EDA (Phase 1)

**Datenbasis:** Snapshot `data/raw` (Crates, Subcrates, `database V2`, Playlists) + Index; Tags (tinytag) und Hash (SHA-256) nur lesend vom Stick.

**Kernergebnisse:**
| Thema | Ergebnis |
|---|---|
| Crates (Wahrheitsquelle) | 7.915 Einträge, 2.754 Pfade; Stufe 1+2 sicher 85,3 %; Stufe 3 (Akzente) 0 |
| Playlist-Ebene | 125/167 Crates (74,9 %) vollständig sicher; Mittel je Crate 93,2 % (KI 90,1–96,0 %) |
| Fehltreffer Crates (404) | a Name 1× anderswo 166; d/e Name nicht im Index 238 |
| Subcrates (2021) | zusätzliche Quelle, Varianten `_2021` (Entscheidung d) |
| `database V2` | TLV, 5.478 Tracks; Abdeckung 96,8 % / 100 %; Titel enthält DJ-Notizen |
| Audio-Tags | 5.236 Dateien, 0 Fehler; Dauer 100 %, Tracknummer nur 61 % |
| Inhaltsbeleg Dauer | 96,6 % ≤ 1 s bei gleicher Datei; 3,2 % mp3-Ausreißer (Ursache offen, Phase 4) |
| Übrige Quellen | 10.289 Einträge; foobar 100 % / 99,1 %, Mac 95,8 %, Musikordner relativ 75,9 %, fremde Windows-Pfade 0 % |
| Namenssuche übrige Quellen | 3.000 Einträge mit genau 1 Namenskandidaten, davon 1.972 mit gleichem Ordner + Datei |
| Dubletten | 45 Paare bitgleich; bei 15/48 Mehrfach-Kandidaten alle bitgleich |
| **0-Byte-„Ordner"** | **10 Album-Ordner sind 0-Byte-Dateien; betroffen 229/238 Crate-Pfade ohne Kandidaten und 509/894 übrige Einträge (Hypothese Dateisystemschaden)** |
| Zielformat | absolut, Laufwerk je Laptop (Asus D:, Medion E:), Zielordner auf dem Stick |

**Antwort auf die Leitfrage (Stand Phase 1):**
1. **Exakter Pfad (Stufe 1+2), deterministisch:** löst 85 % der Crate-Pfade und 75 % der Crates vollständig.
2. **Namenssuche (wie Suche im Explorer), deterministisch:** eindeutiger Name bzw. eindeutiges Pfadende bei 166 Crate-Pfaden und 3.000 Einträgen übriger Quellen. Offen ist nur die Klasse („sicher" oder „Vorschlag") → Precision am Gold-Set.
3. **Datenverlust, kein Verfahren hilft:** 229 Crate-Pfade und 509 Einträge zeigen in 10 Album-„Ordner", die 0-Byte-Dateien sind → Bericht „Album beschädigt/verloren".
4. **Raum für Inhaltsbelege, Fuzzy-, ML- und Agent-Verfahren:** 9 Crate-Pfade und 385 Einträge ohne Namenskandidaten (Suche über Metadaten: Titel, Dauer) sowie 33 Einträge mit verschiedenen Mehrfach-Kandidaten (Auswahl über Inhaltsbelege oder Nutzer).

**Entscheidungen Phase 1:** Subcrates Variante d (09.11.) | tinytag statt mutagen (11.11.) | Dauer-Abweichung kein hartes Veto bis Phase 4 (11.11.) | Zielformat absolut je Laptop (12.11.) | Split gruppiert nach Ordner, Gold-Set mit Pilot (12.11.).

**Empfehlung an den Nutzer:** vollständige Sicherung des Sticks